#Phase 1: Colab Environment Setup
Mount Google Drive (Python)

**Purpose:** Mount Google Drive so all project files under `prion_rnaseq_meta_analysis/` are accessible at `/content/drive/MyDrive/`.

**Inputs:** Google account with the project folder in Drive.

**Outputs:** Drive mounted at `/content/drive`.

**Key decisions:** Run this cell first every session; re-mounting is safe.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##(Python): Enable R

**Purpose:** Load the `rpy2.ipython` extension so subsequent cells can use the `%%R` magic to run R code inside this Python notebook.

**Inputs:** None.

**Outputs:** R magic enabled for all subsequent `%%R` cells.

**Key decisions:** Must be run before any `%%R` cell; safe to re-run.

In [2]:
%load_ext rpy2.ipython

##(R): Section 0 — Package Installation & Loading

**Purpose:** Install and load all R packages required by this notebook. GEOquery (Bioconductor) is used to download GEO SOFT files; readr/dplyr/stringr handle data wrangling; httr/jsonlite support NCBI Entrez API calls.

**Inputs:** Internet connection (first run only; packages cached in Colab session).

**Outputs:** All packages loaded into the R session.

**Key decisions:** `options(timeout = 600)` prevents timeouts on large GEO downloads. GEOquery is installed via BiocManager; CRAN packages via `install.packages()`.

In [3]:
%%R
# =============================================================================
# SECTION 0: PACKAGE INSTALLATION AND LOADING
# =============================================================================

if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

cran_pkgs <- c(
  "GEOquery",   # Parse GEO series matrix files (Bioconductor)
  "readr",      # Fast CSV reading/writing
  "dplyr",      # Data manipulation
  "stringr",    # String manipulation
  "httr",       # HTTP requests to NCBI Entrez API
  "jsonlite"    # Parse JSON from Entrez API
)

# GEOquery is a Bioconductor package; install via BiocManager
if (!requireNamespace("GEOquery", quietly = TRUE))
  BiocManager::install("GEOquery", ask = FALSE)

cran_only <- c("readr", "dplyr", "stringr", "httr", "jsonlite")
for (pkg in cran_only) {
  if (!requireNamespace(pkg, quietly = TRUE))
    install.packages(pkg)
}

suppressPackageStartupMessages({
  library(GEOquery)
  library(readr)
  library(dplyr)
  library(stringr)
  library(httr)
  library(jsonlite)
})

cat("Packages loaded.\n\n")

# Colab-specific fix for large GEO file downloads
options(timeout = 600)

Packages loaded.



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/BiocManager_1.30.27.tar.gz'
Content type 'application/x-gzip' length 752490 bytes (734 KB)
downloaded 734 KB


The downloaded source packages are in
	‘/tmp/RtmpfKV4mr/downloaded_packages’
'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com
Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)
Installing package(s) 'BiocVersion', 'GEOquery'
also installing the dependencies ‘XVector’, ‘matrixStats’, ‘abind’, ‘SparseArray’, ‘BiocGenerics’, ‘statmod’, ‘XML’, ‘R.oo’, ‘R.methodsS3’, ‘MatrixGenerics’, ‘GenomicRanges’, ‘IRanges’, ‘Seqinfo’, ‘S4Arrays’, ‘DelayedArray’, ‘Biobase’, ‘limma’, ‘rentrez’, ‘R.utils’, ‘SummarizedExperiment’, ‘S4Vectors’

trying URL 'https://bioconductor.org/packages/3.23/bioc/src/contrib

##(R): Section 1 — Configuration

**Purpose:** Define all file paths, directory constants, and the study registry used throughout this notebook. Centralising paths here means only this cell needs editing when the project is moved.

**Inputs:** None (constants defined inline).

**Outputs:** `METADATA_TSV`, `OUTPUT_DIR`, `SALMON_DIR`, `GEO_CACHE_DIR`, `NCBI_API_KEY`, `API_SLEEP`, `STUDY_STRATA` available in the R environment.

**Key decisions:** `STUDY_STRATA` maps each GSE to its analysis stratum (A = MCPS mouse brain, B = human brain, C = human validation, D = held-out replication). Set `NCBI_API_KEY` env var to increase NCBI rate limit from 3 to 10 req/s.

In [4]:
%%R
# --- Colab Directory Setup ---
# Change this to match your exact folder structure in Google Drive
setwd("/content/drive/MyDrive/prion_rnaseq_meta_analysis/scripts")

# =============================================================================
# SECTION 1: CONFIGURATION
# =============================================================================

# --- Paths ---
METADATA_TSV  <- "../config/sample_metadata.tsv"
OUTPUT_DIR    <- "../config/coldata"
SALMON_DIR    <- "../results/salmon_quant"
GEO_CACHE_DIR <- "../data/raw/geo_soft"

dir.create(OUTPUT_DIR,    recursive = TRUE, showWarnings = FALSE)
dir.create(GEO_CACHE_DIR, recursive = TRUE, showWarnings = FALSE)

# --- NCBI API key (optional — set env var NCBI_API_KEY to speed up requests) ---
NCBI_API_KEY <- Sys.getenv("NCBI_API_KEY", unset = "")
# Rate limit: 0.35s without key (safe for 3 req/s limit), 0.12s with key
API_SLEEP <- if (nchar(NCBI_API_KEY) > 0) 0.12 else 0.35

cat(sprintf("Output directory : %s\n", OUTPUT_DIR))
cat(sprintf("GEO cache        : %s\n", GEO_CACHE_DIR))
cat(sprintf("NCBI API key     : %s\n", if (nchar(NCBI_API_KEY) > 0) "SET" else "not set (3 req/s limit)"))
cat(sprintf("API sleep        : %.2fs between requests\n\n", API_SLEEP))

# --- Study registry ---
# All 11 studies with their strata
STUDY_STRATA <- c(
  GSE149805 = "A", GSE198063 = "A", GSE144738 = "A",
  GSE189527 = "A", GSE201249 = "A",
  GSE210128 = "B",
  GSE214373 = "C",
  GSE90977  = "D", GSE124932 = "D", GSE207251 = "D", GSE202275 = "D"
)

Output directory : ../config/coldata
GEO cache        : ../data/raw/geo_soft
NCBI API key     : not set (3 req/s limit)
API sleep        : 0.35s between requests



##(R): Section 2 — Load Sample Metadata

**Purpose:** Read `sample_metadata.tsv` — the master sample registry listing all 867 SRR accessions, their parent GSE, library layout (SINGLE/PAIRED), and organism. This file was built manually from NCBI SRA RunInfo.

**Inputs:** `../config/sample_metadata.tsv` (867 rows × ≥4 columns: SRR, GSE, LAYOUT, ORGANISM).

**Outputs:** `sample_meta` data frame in R.

**Key decisions:** Script halts with an informative error if the file is missing or required columns are absent — fail-fast design prevents silent downstream errors.

In [5]:
%%R
# =============================================================================
# SECTION 2: LOAD SAMPLE METADATA
# =============================================================================

cat("Loading sample_metadata.tsv...\n")

if (!file.exists(METADATA_TSV)) {
  stop(sprintf(
    "sample_metadata.tsv not found at: %s\n  This file must exist before running this script.",
    METADATA_TSV
  ))
}

sample_meta <- read_tsv(METADATA_TSV, col_types = cols(.default = "c"), show_col_types = FALSE)
required_cols <- c("SRR", "GSE", "LAYOUT", "ORGANISM")
missing_cols  <- setdiff(required_cols, colnames(sample_meta))
if (length(missing_cols) > 0) {
  stop(sprintf("sample_metadata.tsv is missing columns: %s", paste(missing_cols, collapse = ", ")))
}

cat(sprintf("  Loaded %d SRR entries across %d studies.\n\n",
            nrow(sample_meta), n_distinct(sample_meta$GSE)))

Loading sample_metadata.tsv...
  Loaded 867 SRR entries across 11 studies.



##(R): Section 3 — Helper Functions

**Purpose:** Define reusable functions for GEO data retrieval and parsing: `fetch_geo()` downloads SOFT files with local caching; `parse_characteristics()` extracts key:value metadata from GSM objects; `extract_srx()` pulls SRX accessions from relation fields; `resolve_srr()` maps SRX → SRR via a hard-coded fallback table (867 entries) with NCBI Entrez API as backup.

**Inputs:** None (function definitions only).

**Outputs:** Four functions available in the R environment.

**Key decisions:** The `SRX_SRR_FALLBACK` hard-coded map eliminates NCBI API calls for all 867 known samples, ensuring reproducibility even when NCBI is unavailable.

In [6]:
%%R
# =============================================================================
# SECTION 3: HELPER FUNCTIONS
# =============================================================================

#' Fetch GEO series object via GEOquery with local caching.
#' Uses GSEMatrix=FALSE to download the full SOFT file, which is required to
#' populate Meta(gsm)$characteristics_ch1 and Meta(gsm)$relation for all GSMs.
#' (GSEMatrix=TRUE only downloads the series matrix and does NOT include per-sample
#'  SOFT metadata such as characteristics_ch1 — it would silently return empty lists.)
#' Returns a list with element $gsms (named list of GSM objects) to keep the
#' rest of the script's interface identical to the original GEOparse version.
fetch_geo <- function(gse_id, cache_dir = GEO_CACHE_DIR) {
  cat(sprintf("  Fetching GEO metadata for %s (SOFT file)...\n", gse_id))
  gse_obj <- tryCatch(
    GEOquery::getGEO(gse_id, destdir = cache_dir, GSEMatrix = FALSE,
                     getGPL = FALSE),
    error = function(e) {
      stop(sprintf("Failed to fetch GEO data for %s: %s", gse_id, e$message))
    }
  )
  # With GSEMatrix=FALSE, getGEO() returns a GSE object directly.
  # GSMList() extracts the named list of GSM objects with full Meta() populated.
  gsm_list <- GEOquery::GSMList(gse_obj)
  gse <- list(gsms = gsm_list)
  cat(sprintf("  %s: %d GSM samples found.\n", gse_id, length(gse$gsms)))
  return(gse)
}


#' Extract characteristics from a GSM object into a named list.
#' Handles the "key: value" format used in GEO series matrices.
#' GEOquery: characteristics are in Meta(gsm)$characteristics_ch1
parse_characteristics <- function(gsm) {
  chars <- GEOquery::Meta(gsm)$characteristics_ch1
  if (is.null(chars) || length(chars) == 0) return(list())
  result <- list()
  for (ch in chars) {
    parts <- str_split_fixed(ch, ":\\s*", 2)
    if (ncol(parts) == 2 && nchar(parts[1,1]) > 0) {
      key <- str_trim(str_to_lower(parts[1,1]))
      val <- str_trim(parts[1,2])
      result[[key]] <- val
    }
  }
  return(result)
}


#' Extract SRX accession from GSM relation field.
#' GEOquery: relation field is in Meta(gsm)$relation
extract_srx <- function(gsm) {
  relations <- GEOquery::Meta(gsm)$relation
  if (is.null(relations)) return(NA_character_)
  sra_links <- relations[str_detect(relations, "SRX")]
  if (length(sra_links) == 0) return(NA_character_)
  srx <- str_extract(sra_links[1], "SRX\\d+")
  return(srx)
}


#' Resolve SRX experiment accession to SRR run accession via NCBI Entrez.
#' Returns the first SRR associated with the SRX (most studies have 1:1 mapping).
#' Uses a two-tier lookup:
#'   1. SRX_SRR_FALLBACK: hard-coded map of all 867 SRX->SRR pairs, built from
#'      NCBI SRA RunInfo on 2026-05-15. Eliminates API calls for known samples
#'      and makes the script robust to NCBI API failures or rate limits.
#'   2. NCBI Entrez API: fallback for any SRX not in the hard-coded map.
#' Uses a simple in-memory cache to avoid redundant API calls.
#'
#' SRX_SRR_FALLBACK: complete SRX->SRR map for all 867 samples across 11 studies.
#' Built from NCBI SRA RunInfo API on 2026-05-15. Verified against sample_metadata.tsv.
#' This map eliminates NCBI API calls for all known samples and ensures reproducibility.
SRX_SRR_FALLBACK <- c(
  SRX13237463 = "SRR17047684",
  SRX13237464 = "SRR17047709",
  SRX13237465 = "SRR17047710",
  SRX13237466 = "SRR17047711",
  SRX13237467 = "SRR17047712",
  SRX13237468 = "SRR17047713",
  SRX13237469 = "SRR17047714",
  SRX13237470 = "SRR17047715",
  SRX13237471 = "SRR17047716",
  SRX13237472 = "SRR17047757",
  SRX13237473 = "SRR17047758",
  SRX13237474 = "SRR17047759",
  SRX13237475 = "SRR17047760",
  SRX13237476 = "SRR17047761",
  SRX13237477 = "SRR17047762",
  SRX13237478 = "SRR17047763",
  SRX13237479 = "SRR17047764",
  SRX13237480 = "SRR17047733",
  SRX13237481 = "SRR17047734",
  SRX13237482 = "SRR17047735",
  SRX13237483 = "SRR17047736",
  SRX13237484 = "SRR17047737",
  SRX13237485 = "SRR17047738",
  SRX13237486 = "SRR17047739",
  SRX13237487 = "SRR17047740",
  SRX13237488 = "SRR17047741",
  SRX13237489 = "SRR17047742",
  SRX13237490 = "SRR17047743",
  SRX13237491 = "SRR17047744",
  SRX13237492 = "SRR17047745",
  SRX13237493 = "SRR17047746",
  SRX13237494 = "SRR17047747",
  SRX13237495 = "SRR17047748",
  SRX13237496 = "SRR17047781",
  SRX13237497 = "SRR17047717",
  SRX13237498 = "SRR17047718",
  SRX13237499 = "SRR17047719",
  SRX13237500 = "SRR17047720",
  SRX13237501 = "SRR17047721",
  SRX13237502 = "SRR17047722",
  SRX13237503 = "SRR17047723",
  SRX13237504 = "SRR17047724",
  SRX13237505 = "SRR17047685",
  SRX13237506 = "SRR17047686",
  SRX13237507 = "SRR17047687",
  SRX13237508 = "SRR17047688",
  SRX13237509 = "SRR17047689",
  SRX13237510 = "SRR17047690",
  SRX13237511 = "SRR17047691",
  SRX13237512 = "SRR17047692",
  SRX13237513 = "SRR17047693",
  SRX13237514 = "SRR17047694",
  SRX13237515 = "SRR17047695",
  SRX13237516 = "SRR17047696",
  SRX13237517 = "SRR17047697",
  SRX13237518 = "SRR17047698",
  SRX13237519 = "SRR17047699",
  SRX13237520 = "SRR17047700",
  SRX13237521 = "SRR17047765",
  SRX13237522 = "SRR17047766",
  SRX13237523 = "SRR17047767",
  SRX13237524 = "SRR17047768",
  SRX13237525 = "SRR17047769",
  SRX13237526 = "SRR17047770",
  SRX13237527 = "SRR17047771",
  SRX13237528 = "SRR17047772",
  SRX13237529 = "SRR17047701",
  SRX13237530 = "SRR17047702",
  SRX13237531 = "SRR17047703",
  SRX13237532 = "SRR17047704",
  SRX13237533 = "SRR17047705",
  SRX13237534 = "SRR17047706",
  SRX13237535 = "SRR17047707",
  SRX13237536 = "SRR17047708",
  SRX13237537 = "SRR17047725",
  SRX13237538 = "SRR17047726",
  SRX13237539 = "SRR17047727",
  SRX13237540 = "SRR17047728",
  SRX13237541 = "SRR17047729",
  SRX13237542 = "SRR17047730",
  SRX13237543 = "SRR17047731",
  SRX13237544 = "SRR17047732",
  SRX13237545 = "SRR17047749",
  SRX13237546 = "SRR17047750",
  SRX13237547 = "SRR17047751",
  SRX13237548 = "SRR17047752",
  SRX13237549 = "SRR17047753",
  SRX13237550 = "SRR17047754",
  SRX13237551 = "SRR17047755",
  SRX13237552 = "SRR17047756",
  SRX13237553 = "SRR17047773",
  SRX13237554 = "SRR17047774",
  SRX13237555 = "SRR17047775",
  SRX13237556 = "SRR17047776",
  SRX13237557 = "SRR17047777",
  SRX13237558 = "SRR17047778",
  SRX13237559 = "SRR17047779",
  SRX13237560 = "SRR17047780",
  SRX14406571 = "SRR18268432",
  SRX14406572 = "SRR18268431",
  SRX14406573 = "SRR18268430",
  SRX14406574 = "SRR18268429",
  SRX14406575 = "SRR18268428",
  SRX14406576 = "SRR18268427",
  SRX14406577 = "SRR18268426",
  SRX14406578 = "SRR18268425",
  SRX14406579 = "SRR18268424",
  SRX14406580 = "SRR18268423",
  SRX14406581 = "SRR18268422",
  SRX14406582 = "SRR18268421",
  SRX14406583 = "SRR18268420",
  SRX14406584 = "SRR18268419",
  SRX14406585 = "SRR18268418",
  SRX14406586 = "SRR18268417",
  SRX14406587 = "SRR18268416",
  SRX14406588 = "SRR18268415",
  SRX14406589 = "SRR18268414",
  SRX14406590 = "SRR18268413",
  SRX14406591 = "SRR18268412",
  SRX14406592 = "SRR18268411",
  SRX14406593 = "SRR18268410",
  SRX14406594 = "SRR18268409",
  SRX14406595 = "SRR18268408",
  SRX14406596 = "SRR18268407",
  SRX14406597 = "SRR18268406",
  SRX14406598 = "SRR18268405",
  SRX14406599 = "SRR18268404",
  SRX14406600 = "SRR18268403",
  SRX14406601 = "SRR18268402",
  SRX14406602 = "SRR18268401",
  SRX14406603 = "SRR18268400",
  SRX14406604 = "SRR18268399",
  SRX14406605 = "SRR18268398",
  SRX14406606 = "SRR18268397",
  SRX14406607 = "SRR18268396",
  SRX14406608 = "SRR18268395",
  SRX14406609 = "SRR18268394",
  SRX14406610 = "SRR18268393",
  SRX14406611 = "SRR18268392",
  SRX14406612 = "SRR18268391",
  SRX14406613 = "SRR18268390",
  SRX14406614 = "SRR18268389",
  SRX14406615 = "SRR18268388",
  SRX14406616 = "SRR18268387",
  SRX14406617 = "SRR18268386",
  SRX14406618 = "SRR18268385",
  SRX14406619 = "SRR18268384",
  SRX14406620 = "SRR18268383",
  SRX14406621 = "SRR18268382",
  SRX14406622 = "SRR18268381",
  SRX14406623 = "SRR18268380",
  SRX14406624 = "SRR18268379",
  SRX14406625 = "SRR18268378",
  SRX14406626 = "SRR18268377",
  SRX14406627 = "SRR18268376",
  SRX14406628 = "SRR18268375",
  SRX14406629 = "SRR18268374",
  SRX14406630 = "SRR18268373",
  SRX14406631 = "SRR18268372",
  SRX14406632 = "SRR18268371",
  SRX14406633 = "SRR18268370",
  SRX14406634 = "SRR18268369",
  SRX14406635 = "SRR18268368",
  SRX14406636 = "SRR18268367",
  SRX14406637 = "SRR18268366",
  SRX14406638 = "SRR18268365",
  SRX14406639 = "SRR18268364",
  SRX14406640 = "SRR18268363",
  SRX14406641 = "SRR18268362",
  SRX14406642 = "SRR18268361",
  SRX14406643 = "SRR18268360",
  SRX14406644 = "SRR18268359",
  SRX14406645 = "SRR18268358",
  SRX14406646 = "SRR18268357",
  SRX14406647 = "SRR18268356",
  SRX14406648 = "SRR18268355",
  SRX14406649 = "SRR18268354",
  SRX14406650 = "SRR18268353",
  SRX14406651 = "SRR18268352",
  SRX14406652 = "SRR18268351",
  SRX14406653 = "SRR18268350",
  SRX14406654 = "SRR18268349",
  SRX14406655 = "SRR18268348",
  SRX14406656 = "SRR18268347",
  SRX14406657 = "SRR18268346",
  SRX14406658 = "SRR18268345",
  SRX14406659 = "SRR18268344",
  SRX14406660 = "SRR18268343",
  SRX14406661 = "SRR18268342",
  SRX14406662 = "SRR18268341",
  SRX14406663 = "SRR18268340",
  SRX14406664 = "SRR18268339",
  SRX14406665 = "SRR18268338",
  SRX14406666 = "SRR18268337",
  SRX14406667 = "SRR18268336",
  SRX14406668 = "SRR18268335",
  SRX14406669 = "SRR18268334",
  SRX14406670 = "SRR18268333",
  SRX14406671 = "SRR18268332",
  SRX14406672 = "SRR18268331",
  SRX14406673 = "SRR18268330",
  SRX14406674 = "SRR18268329",
  SRX14406675 = "SRR18268328",
  SRX14406676 = "SRR18268327",
  SRX14406677 = "SRR18268326",
  SRX14406678 = "SRR18268325",
  SRX14406679 = "SRR18268324",
  SRX14406680 = "SRR18268323",
  SRX14406681 = "SRR18268322",
  SRX14406682 = "SRR18268321",
  SRX14406683 = "SRR18268320",
  SRX14406684 = "SRR18268319",
  SRX14406685 = "SRR18268318",
  SRX14406686 = "SRR18268317",
  SRX14406687 = "SRR18268316",
  SRX14406688 = "SRR18268315",
  SRX14406689 = "SRR18268314",
  SRX14406690 = "SRR18268313",
  SRX14406691 = "SRR18268312",
  SRX14406692 = "SRR18268311",
  SRX14406693 = "SRR18268310",
  SRX14406694 = "SRR18268309",
  SRX14406695 = "SRR18268308",
  SRX14406696 = "SRR18268307",
  SRX14406697 = "SRR18268306",
  SRX14406698 = "SRR18268305",
  SRX14406699 = "SRR18268304",
  SRX14406700 = "SRR18268303",
  SRX14406701 = "SRR18268302",
  SRX14406702 = "SRR18268301",
  SRX14406703 = "SRR18268300",
  SRX14406704 = "SRR18268299",
  SRX14406705 = "SRR18268298",
  SRX14406706 = "SRR18268297",
  SRX14406707 = "SRR18268296",
  SRX14406708 = "SRR18268295",
  SRX14406709 = "SRR18268294",
  SRX14406710 = "SRR18268293",
  SRX14406711 = "SRR18268292",
  SRX14406712 = "SRR18268291",
  SRX14406713 = "SRR18268290",
  SRX14406714 = "SRR18268289",
  SRX14406715 = "SRR18268288",
  SRX14406716 = "SRR18268287",
  SRX14406717 = "SRR18268286",
  SRX14406718 = "SRR18268285",
  SRX14406719 = "SRR18268284",
  SRX14406720 = "SRR18268283",
  SRX14406721 = "SRR18268282",
  SRX14406722 = "SRR18268281",
  SRX14406723 = "SRR18268280",
  SRX14406724 = "SRR18268279",
  SRX14406725 = "SRR18268278",
  SRX14406726 = "SRR18268277",
  SRX14406727 = "SRR18268276",
  SRX14406728 = "SRR18268275",
  SRX14406729 = "SRR18268274",
  SRX14956282 = "SRR18858544",
  SRX14956283 = "SRR18858543",
  SRX14956284 = "SRR18858542",
  SRX14956285 = "SRR18858541",
  SRX14956286 = "SRR18858540",
  SRX14956287 = "SRR18858539",
  SRX14956288 = "SRR18858538",
  SRX14956289 = "SRR18858537",
  SRX14956290 = "SRR18858536",
  SRX14956291 = "SRR18858535",
  SRX14956292 = "SRR18858534",
  SRX14956293 = "SRR18858533",
  SRX14956294 = "SRR18858532",
  SRX14956295 = "SRR18858531",
  SRX14956296 = "SRR18858530",
  SRX14956297 = "SRR18858529",
  SRX14956298 = "SRR18858528",
  SRX14956299 = "SRR18858527",
  SRX14956300 = "SRR18858526",
  SRX14956301 = "SRR18858525",
  SRX14956302 = "SRR18858524",
  SRX14956303 = "SRR18858523",
  SRX14956304 = "SRR18858522",
  SRX14956305 = "SRR18858521",
  SRX14956306 = "SRR18858520",
  SRX14956307 = "SRR18858519",
  SRX14956308 = "SRR18858518",
  SRX14956309 = "SRR18858517",
  SRX14956310 = "SRR18858516",
  SRX14956311 = "SRR18858515",
  SRX14956312 = "SRR18858514",
  SRX14956313 = "SRR18858513",
  SRX14956314 = "SRR18858512",
  SRX14956315 = "SRR18858511",
  SRX14956316 = "SRR18858510",
  SRX14956317 = "SRR18858509",
  SRX14956318 = "SRR18858508",
  SRX14956319 = "SRR18858507",
  SRX14956320 = "SRR18858506",
  SRX14956321 = "SRR18858505",
  SRX14956322 = "SRR18858504",
  SRX14956323 = "SRR18858503",
  SRX14956324 = "SRR18858502",
  SRX14956325 = "SRR18858501",
  SRX14956326 = "SRR18858500",
  SRX14956327 = "SRR18858499",
  SRX14956328 = "SRR18858498",
  SRX14956329 = "SRR18858497",
  SRX14956330 = "SRR18858496",
  SRX14956331 = "SRR18858495",
  SRX15161542 = "SRR19091884",
  SRX15161543 = "SRR19091883",
  SRX15161544 = "SRR19091882",
  SRX15161545 = "SRR19091881",
  SRX15161546 = "SRR19091880",
  SRX15161547 = "SRR19091879",
  SRX15161548 = "SRR19091878",
  SRX15161549 = "SRR19091877",
  SRX15161550 = "SRR19091876",
  SRX15161551 = "SRR19091875",
  SRX15161552 = "SRR19091874",
  SRX15161553 = "SRR19091873",
  SRX15950427 = "SRR19907652",
  SRX15950428 = "SRR19907651",
  SRX15950429 = "SRR19907650",
  SRX15950430 = "SRR19907649",
  SRX15950431 = "SRR19907648",
  SRX15950432 = "SRR19907647",
  SRX16741376 = "SRR20720719",
  SRX16741377 = "SRR20720720",
  SRX16741378 = "SRR20720721",
  SRX16741379 = "SRR20720722",
  SRX16741380 = "SRR20720723",
  SRX16741381 = "SRR20720724",
  SRX16741382 = "SRR20720725",
  SRX16741383 = "SRR20720726",
  SRX16741384 = "SRR20720727",
  SRX16741388 = "SRR20720731",
  SRX16741389 = "SRR20720732",
  SRX16741390 = "SRR20720733",
  SRX16741394 = "SRR20720769",
  SRX16741395 = "SRR20720770",
  SRX16741396 = "SRR20720771",
  SRX16741397 = "SRR20720772",
  SRX16741398 = "SRR20720773",
  SRX16741402 = "SRR20720777",
  SRX16741403 = "SRR20720778",
  SRX16741404 = "SRR20720779",
  SRX16741408 = "SRR20720783",
  SRX16741409 = "SRR20720784",
  SRX16741410 = "SRR20720833",
  SRX16741411 = "SRR20720834",
  SRX16741412 = "SRR20720835",
  SRX16741413 = "SRR20720889",
  SRX16741414 = "SRR20720890",
  SRX16741415 = "SRR20720891",
  SRX16741416 = "SRR20720892",
  SRX16741417 = "SRR20720893",
  SRX16741418 = "SRR20720894",
  SRX16741419 = "SRR20720895",
  SRX16741420 = "SRR20720896",
  SRX16741421 = "SRR20720897",
  SRX16741422 = "SRR20720898",
  SRX16741423 = "SRR20720899",
  SRX16741424 = "SRR20720900",
  SRX16741425 = "SRR20720901",
  SRX16741426 = "SRR20720950",
  SRX16741427 = "SRR20720951",
  SRX16741428 = "SRR20720952",
  SRX16741429 = "SRR20720953",
  SRX16741430 = "SRR20720954",
  SRX16741431 = "SRR20720955",
  SRX16741432 = "SRR20720956",
  SRX16741433 = "SRR20720957",
  SRX16741434 = "SRR20720958",
  SRX16741435 = "SRR20720959",
  SRX16741436 = "SRR20720960",
  SRX16741437 = "SRR20720961",
  SRX16741438 = "SRR20720962",
  SRX16741439 = "SRR20720963",
  SRX16741440 = "SRR20720964",
  SRX16741441 = "SRR20720965",
  SRX16741442 = "SRR20720737",
  SRX16741443 = "SRR20720738",
  SRX16741447 = "SRR20720742",
  SRX16741448 = "SRR20720743",
  SRX16741449 = "SRR20720744",
  SRX16741450 = "SRR20720745",
  SRX16741453 = "SRR20720748",
  SRX16741454 = "SRR20720749",
  SRX16741455 = "SRR20720750",
  SRX16741456 = "SRR20720751",
  SRX16741457 = "SRR20720752",
  SRX16741458 = "SRR20720785",
  SRX16741462 = "SRR20720789",
  SRX16741463 = "SRR20720790",
  SRX16741464 = "SRR20720791",
  SRX16741468 = "SRR20720795",
  SRX16741469 = "SRR20720796",
  SRX16741470 = "SRR20720797",
  SRX16741471 = "SRR20720798",
  SRX16741472 = "SRR20720799",
  SRX16741473 = "SRR20720800",
  SRX16741474 = "SRR20720902",
  SRX16741475 = "SRR20720903",
  SRX16741476 = "SRR20720904",
  SRX16741477 = "SRR20720905",
  SRX16741478 = "SRR20720906",
  SRX16741479 = "SRR20720907",
  SRX16741480 = "SRR20720908",
  SRX16741481 = "SRR20720909",
  SRX16741482 = "SRR20720910",
  SRX16741483 = "SRR20720911",
  SRX16741484 = "SRR20720912",
  SRX16741485 = "SRR20720913",
  SRX16741486 = "SRR20720914",
  SRX16741487 = "SRR20720915",
  SRX16741488 = "SRR20720916",
  SRX16741489 = "SRR20720917",
  SRX16741490 = "SRR20720966",
  SRX16741491 = "SRR20720967",
  SRX16741492 = "SRR20720968",
  SRX16741493 = "SRR20720969",
  SRX16741494 = "SRR20720970",
  SRX16741495 = "SRR20720971",
  SRX16741496 = "SRR20720972",
  SRX16741497 = "SRR20720973",
  SRX16741538 = "SRR20720753",
  SRX16741539 = "SRR20720754",
  SRX16741540 = "SRR20720755",
  SRX16741541 = "SRR20720756",
  SRX16741544 = "SRR20720759",
  SRX16741545 = "SRR20720760",
  SRX16741546 = "SRR20720761",
  SRX16741550 = "SRR20720765",
  SRX16741551 = "SRR20720766",
  SRX16741552 = "SRR20720767",
  SRX16741553 = "SRR20720768",
  SRX16741554 = "SRR20720801",
  SRX16741555 = "SRR20720802",
  SRX16741556 = "SRR20720803",
  SRX16741557 = "SRR20720804",
  SRX16741558 = "SRR20720805",
  SRX16741559 = "SRR20720806",
  SRX16741560 = "SRR20720807",
  SRX16741561 = "SRR20720808",
  SRX16741562 = "SRR20720809",
  SRX16741563 = "SRR20720810",
  SRX16741564 = "SRR20720811",
  SRX16741565 = "SRR20720812",
  SRX16741566 = "SRR20720813",
  SRX16741567 = "SRR20720814",
  SRX16741568 = "SRR20720815",
  SRX16741569 = "SRR20720816",
  SRX16741570 = "SRR20720918",
  SRX16741571 = "SRR20720919",
  SRX16741572 = "SRR20720920",
  SRX16741573 = "SRR20720921",
  SRX16741574 = "SRR20720922",
  SRX16741576 = "SRR20720924",
  SRX16741577 = "SRR20720925",
  SRX16741578 = "SRR20720926",
  SRX16741579 = "SRR20720927",
  SRX16741580 = "SRR20720928",
  SRX16741581 = "SRR20720929",
  SRX16741582 = "SRR20720930",
  SRX16741583 = "SRR20720931",
  SRX16741584 = "SRR20720932",
  SRX16741585 = "SRR20720933",
  SRX16741618 = "SRR20720817",
  SRX16741619 = "SRR20720818",
  SRX16741620 = "SRR20720819",
  SRX16741621 = "SRR20720820",
  SRX16741622 = "SRR20720821",
  SRX16741623 = "SRR20720822",
  SRX16741624 = "SRR20720823",
  SRX16741625 = "SRR20720824",
  SRX16741626 = "SRR20720825",
  SRX16741627 = "SRR20720826",
  SRX16741628 = "SRR20720827",
  SRX16741629 = "SRR20720828",
  SRX16741630 = "SRR20720829",
  SRX16741631 = "SRR20720830",
  SRX16741632 = "SRR20720831",
  SRX16741633 = "SRR20720832",
  SRX16741634 = "SRR20720934",
  SRX16741635 = "SRR20720935",
  SRX16741636 = "SRR20720936",
  SRX16741637 = "SRR20720937",
  SRX16741638 = "SRR20720938",
  SRX16741639 = "SRR20720939",
  SRX16741640 = "SRR20720940",
  SRX16741641 = "SRR20720941",
  SRX16741642 = "SRR20720942",
  SRX16741643 = "SRR20720943",
  SRX16741644 = "SRR20720944",
  SRX16741645 = "SRR20720945",
  SRX16741646 = "SRR20720946",
  SRX16741647 = "SRR20720947",
  SRX16741648 = "SRR20720948",
  SRX16741649 = "SRR20720949",
  SRX17737750 = "SRR21742178",
  SRX17737751 = "SRR21742177",
  SRX17737752 = "SRR21742174",
  SRX17737753 = "SRR21742173",
  SRX17737754 = "SRR21742175",
  SRX17737755 = "SRR21742172",
  SRX17737756 = "SRR21742176",
  SRX17737757 = "SRR21742171",
  SRX17737758 = "SRR21742170",
  SRX17737759 = "SRR21742167",
  SRX17737760 = "SRR21742166",
  SRX17737761 = "SRR21742168",
  SRX17737762 = "SRR21742165",
  SRX17737763 = "SRR21742169",
  SRX2401950 = "SRR5084283",
  SRX2401951 = "SRR5084284",
  SRX2401952 = "SRR5084285",
  SRX2401953 = "SRR5084286",
  SRX2401954 = "SRR5084287",
  SRX2401955 = "SRR5084288",
  SRX2401956 = "SRR5084289",
  SRX2401957 = "SRR5084290",
  SRX2401958 = "SRR5084291",
  SRX2401959 = "SRR5084292",
  SRX2401960 = "SRR5084293",
  SRX2401961 = "SRR5084294",
  SRX2401962 = "SRR5084295",
  SRX2401963 = "SRR5084296",
  SRX2401964 = "SRR5084297",
  SRX2401965 = "SRR5084298",
  SRX5237654 = "SRR8429837",
  SRX5237655 = "SRR8429838",
  SRX5237656 = "SRR8429839",
  SRX5237657 = "SRR8429840",
  SRX5237658 = "SRR8429841",
  SRX5237659 = "SRR8429842",
  SRX5237660 = "SRR8429843",
  SRX5237661 = "SRR8429844",
  SRX5237662 = "SRR8429845",
  SRX5237663 = "SRR8429846",
  SRX5237664 = "SRR8429847",
  SRX5237665 = "SRR8429848",
  SRX5237666 = "SRR8429849",
  SRX5237667 = "SRR8429850",
  SRX5237668 = "SRR8429851",
  SRX5237669 = "SRR8429852",
  SRX7672891 = "SRR11017791",
  SRX7672892 = "SRR11017792",
  SRX7672893 = "SRR11017793",
  SRX7672894 = "SRR11017794",
  SRX7672895 = "SRR11017795",
  SRX7672896 = "SRR11017796",
  SRX7672897 = "SRR11017797",
  SRX7672898 = "SRR11017798",
  SRX7672899 = "SRR11017799",
  SRX7672900 = "SRR11017800",
  SRX7672901 = "SRR11017801",
  SRX7672902 = "SRR11017802",
  SRX7672903 = "SRR11017803",
  SRX7672904 = "SRR11017804",
  SRX7672905 = "SRR11017805",
  SRX7672906 = "SRR11017806",
  SRX7672907 = "SRR11017807",
  SRX7672908 = "SRR11017808",
  SRX7672909 = "SRR11017809",
  SRX7672910 = "SRR11017810",
  SRX7672911 = "SRR11017811",
  SRX7672912 = "SRR11017812",
  SRX7672913 = "SRR11017813",
  SRX7672914 = "SRR11017814",
  SRX7672915 = "SRR11017815",
  SRX7672916 = "SRR11017816",
  SRX7672917 = "SRR11017817",
  SRX7672918 = "SRR11017818",
  SRX7672919 = "SRR11017819",
  SRX7672920 = "SRR11017820",
  SRX7672921 = "SRR11017821",
  SRX7672922 = "SRR11017822",
  SRX7672923 = "SRR11017823",
  SRX7672924 = "SRR11017824",
  SRX7672925 = "SRR11017825",
  SRX7672926 = "SRR11017826",
  SRX7672927 = "SRR11017827",
  SRX7672928 = "SRR11017828",
  SRX7672929 = "SRR11017829",
  SRX7672930 = "SRR11017831",
  SRX7672931 = "SRR11017832",
  SRX7672932 = "SRR11017833",
  SRX7672933 = "SRR11017834",
  SRX7672934 = "SRR11017835",
  SRX7672935 = "SRR11017836",
  SRX7672936 = "SRR11017837",
  SRX7672937 = "SRR11017838",
  SRX7672938 = "SRR11017839",
  SRX7672939 = "SRR11017840",
  SRX7672940 = "SRR11017841",
  SRX7672941 = "SRR11017842",
  SRX7672942 = "SRR11017843",
  SRX7672943 = "SRR11017844",
  SRX7672944 = "SRR11017845",
  SRX7672945 = "SRR11017846",
  SRX7672946 = "SRR11017847",
  SRX7672947 = "SRR11017848",
  SRX7672948 = "SRR11017849",
  SRX7672949 = "SRR11017850",
  SRX7672950 = "SRR11017851",
  SRX7672951 = "SRR11017852",
  SRX7672952 = "SRR11017853",
  SRX7672953 = "SRR11017854",
  SRX7672954 = "SRR11017855",
  SRX7672955 = "SRR11017856",
  SRX7673004 = "SRR11017857",
  SRX7673005 = "SRR11017858",
  SRX7673006 = "SRR11017859",
  SRX7673007 = "SRR11017860",
  SRX7673008 = "SRR11017861",
  SRX7673009 = "SRR11017862",
  SRX7673010 = "SRR11017863",
  SRX7673011 = "SRR11017864",
  SRX7673012 = "SRR11017865",
  SRX7673013 = "SRR11017866",
  SRX7673014 = "SRR11017867",
  SRX7673015 = "SRR11017868",
  SRX7673016 = "SRR11017869",
  SRX7673017 = "SRR11017870",
  SRX7673018 = "SRR11017871",
  SRX7673019 = "SRR11017872",
  SRX7673020 = "SRR11017873",
  SRX7673021 = "SRR11017874",
  SRX7673022 = "SRR11017875",
  SRX7673023 = "SRR11017876",
  SRX7673024 = "SRR11017877",
  SRX7673025 = "SRR11017878",
  SRX7673026 = "SRR11017879",
  SRX7673027 = "SRR11017880",
  SRX7673028 = "SRR11017881",
  SRX7673029 = "SRR11017882",
  SRX7673030 = "SRR11017883",
  SRX7673031 = "SRR11017884",
  SRX7673032 = "SRR11017886",
  SRX7673033 = "SRR11017887",
  SRX7673034 = "SRR11017888",
  SRX7673035 = "SRR11017889",
  SRX7673036 = "SRR11017890",
  SRX7673037 = "SRR11017891",
  SRX7673038 = "SRR11017892",
  SRX7673039 = "SRR11017893",
  SRX7673040 = "SRR11017894",
  SRX7673041 = "SRR11017895",
  SRX7673042 = "SRR11017896",
  SRX7673043 = "SRR11017897",
  SRX7673044 = "SRR11017898",
  SRX7673045 = "SRR11017899",
  SRX7673046 = "SRR11017900",
  SRX7673047 = "SRR11017901",
  SRX7673048 = "SRR11017902",
  SRX7673049 = "SRR11017903",
  SRX7673050 = "SRR11017904",
  SRX7673051 = "SRR11017905",
  SRX8245019 = "SRR11684230",
  SRX8245020 = "SRR11684231",
  SRX8245021 = "SRR11684232",
  SRX8245022 = "SRR11684233",
  SRX8245023 = "SRR11684234",
  SRX8245024 = "SRR11684235",
  SRX8245025 = "SRR11684236",
  SRX8245026 = "SRR11684237",
  SRX8245027 = "SRR11684238",
  SRX8245028 = "SRR11684239",
  SRX8245029 = "SRR11684240",
  SRX8245030 = "SRR11684241",
  SRX8245031 = "SRR11684242",
  SRX8245032 = "SRR11684243",
  SRX8245033 = "SRR11684244",
  SRX8245034 = "SRR11684245",
  SRX8245035 = "SRR11684246",
  SRX8245036 = "SRR11684247",
  SRX8245037 = "SRR11684248",
  SRX8245038 = "SRR11684249",
  SRX8245039 = "SRR11684250",
  SRX8245040 = "SRR11684251",
  SRX8245041 = "SRR11684252",
  SRX8245042 = "SRR11684253",
  SRX8245043 = "SRR11684254",
  SRX8245044 = "SRR11684255",
  SRX8245045 = "SRR11684256",
  SRX8245046 = "SRR11684257",
  SRX8245047 = "SRR11684258",
  SRX8245048 = "SRR11684259",
  SRX8245049 = "SRR11684260",
  SRX8245050 = "SRR11684261",
  SRX8245051 = "SRR11684262",
  SRX8245052 = "SRR11684263",
  SRX8245053 = "SRR11684264",
  SRX8245054 = "SRR11684265",
  SRX8245055 = "SRR11684266",
  SRX8245056 = "SRR11684267",
  SRX8245057 = "SRR11684268",
  SRX8245058 = "SRR11684269",
  SRX8245059 = "SRR11684270",
  SRX8245060 = "SRR11684271",
  SRX8245061 = "SRR11684272",
  SRX8245062 = "SRR11684273",
  SRX8245063 = "SRR11684274",
  SRX8245064 = "SRR11684275",
  SRX8245065 = "SRR11684276",
  SRX8245066 = "SRR11684277",
  SRX8245067 = "SRR11684374",
  SRX8245068 = "SRR11684375",
  SRX8245069 = "SRR11684376",
  SRX8245070 = "SRR11684377",
  SRX8245071 = "SRR11684378",
  SRX8245072 = "SRR11684379",
  SRX8245073 = "SRR11684380",
  SRX8245074 = "SRR11684381",
  SRX8245075 = "SRR11684382",
  SRX8245076 = "SRR11684383",
  SRX8245077 = "SRR11684384",
  SRX8245078 = "SRR11684385",
  SRX8245079 = "SRR11684386",
  SRX8245080 = "SRR11684387",
  SRX8245081 = "SRR11684388",
  SRX8245082 = "SRR11684389",
  SRX8245083 = "SRR11684390",
  SRX8245084 = "SRR11684391",
  SRX8245085 = "SRR11684392",
  SRX8245086 = "SRR11684393",
  SRX8245087 = "SRR11684394",
  SRX8245088 = "SRR11684395",
  SRX8245089 = "SRR11684396",
  SRX8245090 = "SRR11684397",
  SRX8245091 = "SRR11684278",
  SRX8245092 = "SRR11684279",
  SRX8245093 = "SRR11684280",
  SRX8245094 = "SRR11684281",
  SRX8245095 = "SRR11684282",
  SRX8245096 = "SRR11684283",
  SRX8245097 = "SRR11684284",
  SRX8245098 = "SRR11684285",
  SRX8245099 = "SRR11684286",
  SRX8245100 = "SRR11684287",
  SRX8245101 = "SRR11684288",
  SRX8245102 = "SRR11684289",
  SRX8245103 = "SRR11684290",
  SRX8245104 = "SRR11684291",
  SRX8245105 = "SRR11684292",
  SRX8245106 = "SRR11684293",
  SRX8245107 = "SRR11684294",
  SRX8245108 = "SRR11684295",
  SRX8245109 = "SRR11684296",
  SRX8245110 = "SRR11684297",
  SRX8245111 = "SRR11684298",
  SRX8245112 = "SRR11684299",
  SRX8245113 = "SRR11684300",
  SRX8245114 = "SRR11684301",
  SRX8245115 = "SRR11684398",
  SRX8245116 = "SRR11684399",
  SRX8245117 = "SRR11684400",
  SRX8245118 = "SRR11684401",
  SRX8245119 = "SRR11684402",
  SRX8245120 = "SRR11684403",
  SRX8245121 = "SRR11684404",
  SRX8245122 = "SRR11684405",
  SRX8245123 = "SRR11684406",
  SRX8245124 = "SRR11684407",
  SRX8245125 = "SRR11684408",
  SRX8245126 = "SRR11684409",
  SRX8245127 = "SRR11684410",
  SRX8245128 = "SRR11684411",
  SRX8245129 = "SRR11684412",
  SRX8245130 = "SRR11684413",
  SRX8245131 = "SRR11684414",
  SRX8245132 = "SRR11684415",
  SRX8245133 = "SRR11684302",
  SRX8245134 = "SRR11684303",
  SRX8245135 = "SRR11684304",
  SRX8245136 = "SRR11684305",
  SRX8245137 = "SRR11684306",
  SRX8245138 = "SRR11684307",
  SRX8245139 = "SRR11684308",
  SRX8245140 = "SRR11684309",
  SRX8245141 = "SRR11684310",
  SRX8245142 = "SRR11684311",
  SRX8245143 = "SRR11684312",
  SRX8245144 = "SRR11684313",
  SRX8245145 = "SRR11684314",
  SRX8245146 = "SRR11684315",
  SRX8245147 = "SRR11684316",
  SRX8245148 = "SRR11684317",
  SRX8245149 = "SRR11684318",
  SRX8245150 = "SRR11684319",
  SRX8245151 = "SRR11684320",
  SRX8245152 = "SRR11684321",
  SRX8245153 = "SRR11684322",
  SRX8245154 = "SRR11684323",
  SRX8245155 = "SRR11684324",
  SRX8245156 = "SRR11684325",
  SRX8245157 = "SRR11684326",
  SRX8245158 = "SRR11684327",
  SRX8245159 = "SRR11684328",
  SRX8245160 = "SRR11684329",
  SRX8245161 = "SRR11684330",
  SRX8245162 = "SRR11684331",
  SRX8245163 = "SRR11684332",
  SRX8245164 = "SRR11684333",
  SRX8245165 = "SRR11684334",
  SRX8245166 = "SRR11684335",
  SRX8245167 = "SRR11684336",
  SRX8245168 = "SRR11684337",
  SRX8245169 = "SRR11684338",
  SRX8245170 = "SRR11684339",
  SRX8245171 = "SRR11684340",
  SRX8245172 = "SRR11684341",
  SRX8245173 = "SRR11684342",
  SRX8245174 = "SRR11684343",
  SRX8245175 = "SRR11684344",
  SRX8245176 = "SRR11684345",
  SRX8245177 = "SRR11684346",
  SRX8245178 = "SRR11684347",
  SRX8245179 = "SRR11684348",
  SRX8245180 = "SRR11684349",
  SRX8245181 = "SRR11684350",
  SRX8245182 = "SRR11684351",
  SRX8245183 = "SRR11684352",
  SRX8245184 = "SRR11684353",
  SRX8245185 = "SRR11684354",
  SRX8245186 = "SRR11684355",
  SRX8245187 = "SRR11684356",
  SRX8245188 = "SRR11684357",
  SRX8245189 = "SRR11684358",
  SRX8245190 = "SRR11684359",
  SRX8245191 = "SRR11684360",
  SRX8245192 = "SRR11684361",
  SRX8245193 = "SRR11684362",
  SRX8245194 = "SRR11684363",
  SRX8245195 = "SRR11684364",
  SRX8245196 = "SRR11684365",
  SRX8245197 = "SRR11684366",
  SRX8245198 = "SRR11684367",
  SRX8245199 = "SRR11684368",
  SRX8245200 = "SRR11684369",
  SRX8245201 = "SRR11684370",
  SRX8245202 = "SRR11684371",
  SRX8245203 = "SRR11684372",
  SRX8245204 = "SRR11684373",
  SRX8533228 = "SRR11999701",
  SRX8533229 = "SRR11999702",
  SRX8533230 = "SRR11999703",
  SRX8533231 = "SRR11999704",
  SRX8533232 = "SRR11999705",
  SRX8533233 = "SRR11999706",
  SRX8533234 = "SRR11999707",
  SRX8533235 = "SRR11999708",
  SRX8533236 = "SRR11999709",
  SRX8533237 = "SRR11999710",
  SRX8533238 = "SRR11999711",
  SRX8533239 = "SRR11999712",
  SRX8533240 = "SRR11999713",
  SRX8533241 = "SRR11999714",
  SRX8533242 = "SRR11999691",
  SRX8533243 = "SRR11999692",
  SRX8533244 = "SRR11999693",
  SRX8533245 = "SRR11999694",
  SRX8533246 = "SRR11999695",
  SRX8533247 = "SRR11999696",
  SRX8533248 = "SRR11999697",
  SRX8533249 = "SRR11999698",
  SRX8533250 = "SRR11999699",
  SRX8533251 = "SRR11999700"
)

srx_cache <- new.env(hash = TRUE, parent = emptyenv())

srx_to_srr <- function(srx_id) {
  if (is.na(srx_id) || nchar(srx_id) == 0) return(NA_character_)

  # Tier 1: check hard-coded fallback map (covers all 867 known samples)
  if (srx_id %in% names(SRX_SRR_FALLBACK)) {
    return(SRX_SRR_FALLBACK[[srx_id]])
  }

  # Tier 2: check in-memory cache (for any SRX resolved via API in this session)
  if (exists(srx_id, envir = srx_cache)) {
    return(get(srx_id, envir = srx_cache))
  }

  Sys.sleep(API_SLEEP)

  # Step 1: esearch to get internal SRA UID
  api_key_param <- if (nchar(NCBI_API_KEY) > 0) paste0("&api_key=", NCBI_API_KEY) else ""
  search_url <- paste0(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
    "?db=sra&term=", srx_id, "&retmax=5&retmode=json", api_key_param
  )

  search_resp <- tryCatch(
    GET(search_url, timeout(15)),
    error = function(e) NULL
  )
  if (is.null(search_resp) || status_code(search_resp) != 200) {
    warning(sprintf("esearch failed for %s", srx_id))
    return(NA_character_)
  }

  search_data <- fromJSON(content(search_resp, "text", encoding = "UTF-8"))
  uids <- search_data$esearchresult$idlist
  if (length(uids) == 0) {
    warning(sprintf("No SRA UID found for %s", srx_id))
    return(NA_character_)
  }

  Sys.sleep(API_SLEEP)

  # Step 2: efetch RunInfo to get SRR
  fetch_url <- paste0(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
    "?db=sra&id=", uids[1], "&rettype=runinfo&retmode=text", api_key_param
  )

  fetch_resp <- tryCatch(
    GET(fetch_url, timeout(15)),
    error = function(e) NULL
  )
  if (is.null(fetch_resp) || status_code(fetch_resp) != 200) {
    warning(sprintf("efetch failed for %s (UID %s)", srx_id, uids[1]))
    return(NA_character_)
  }

  content_text <- content(fetch_resp, "text", encoding = "UTF-8")
  lines <- str_split(str_trim(content_text), "\n")[[1]]
  lines <- lines[nchar(lines) > 0]

  if (length(lines) < 2) {
    warning(sprintf("RunInfo empty for %s", srx_id))
    return(NA_character_)
  }

  header <- str_split(lines[1], ",")[[1]]
  values <- str_split(lines[2], ",")[[1]]
  run_row <- setNames(as.list(values), header)
  srr <- run_row[["Run"]]

  if (is.null(srr) || is.na(srr) || nchar(srr) == 0) {
    warning(sprintf("No Run field in RunInfo for %s", srx_id))
    return(NA_character_)
  }

  # Cache the result
  assign(srx_id, srr, envir = srx_cache)
  return(srr)
}


#' Build the quant.sf file path for a given SRR and GSE.
quant_path <- function(srr, gse, salmon_dir = SALMON_DIR) {
  file.path(salmon_dir, gse, srr, "quant.sf")
}


#' Write a coldata CSV and print a summary.
write_coldata <- function(df, gse_id, out_dir = OUTPUT_DIR) {
  out_path <- file.path(out_dir, paste0(gse_id, "_coldata.csv"))
  write_csv(df, out_path)
  cat(sprintf("  [%s] Written: %s (%d samples)\n", gse_id, out_path, nrow(df)))
  cat(sprintf("  [%s] Condition counts: %s\n", gse_id,
              paste(names(table(df$condition)), table(df$condition), sep = "=", collapse = ", ")))
  invisible(df)
}


#' Build GSM->SRR map for a study by resolving SRX->SRR via Entrez.
#' Also cross-checks against sample_metadata.tsv (which has the ground-truth SRRs).
#' Returns a data.frame with columns: gsm_id, srx_id, srr_id.
build_gsm_srr_map <- function(gse_obj, gse_id) {
  cat(sprintf("  [%s] Resolving GSM -> SRX -> SRR for %d samples...\n",
              gse_id, length(gse_obj$gsms)))

  # Get the SRRs we already know about from sample_metadata.tsv
  known_srrs <- sample_meta %>%
    filter(GSE == gse_id) %>%
    pull(SRR)

  rows <- lapply(names(gse_obj$gsms), function(gsm_id) {
    gsm <- gse_obj$gsms[[gsm_id]]
    srx <- extract_srx(gsm)
    data.frame(gsm_id = gsm_id, srx_id = srx, stringsAsFactors = FALSE)
  })
  gsm_srx_df <- bind_rows(rows)

  # Resolve SRX -> SRR
  cat(sprintf("  [%s] Resolving %d SRX accessions via NCBI Entrez...\n",
              gse_id, sum(!is.na(gsm_srx_df$srx_id))))

  gsm_srx_df$srr_id <- sapply(gsm_srx_df$srx_id, function(srx) {
    if (is.na(srx)) return(NA_character_)
    srr <- srx_to_srr(srx)
    return(srr)
  })

  # Validate: all resolved SRRs should be in sample_metadata.tsv
  resolved_srrs <- gsm_srx_df$srr_id[!is.na(gsm_srx_df$srr_id)]
  missing_from_meta <- setdiff(resolved_srrs, known_srrs)
  missing_from_geo  <- setdiff(known_srrs, resolved_srrs)

  if (length(missing_from_meta) > 0) {
    warning(sprintf(
      "[%s] %d SRRs resolved from GEO but NOT in sample_metadata.tsv: %s",
      gse_id, length(missing_from_meta),
      paste(head(missing_from_meta, 5), collapse = ", ")
    ))
  }
  if (length(missing_from_geo) > 0) {
    warning(sprintf(
      "[%s] %d SRRs in sample_metadata.tsv but NOT resolved from GEO: %s",
      gse_id, length(missing_from_geo),
      paste(head(missing_from_geo, 5), collapse = ", ")
    ))
  }

  n_resolved <- sum(!is.na(gsm_srx_df$srr_id))
  cat(sprintf("  [%s] Resolved %d/%d SRX->SRR mappings.\n",
              gse_id, n_resolved, nrow(gsm_srx_df)))

  return(gsm_srx_df)
}

cat("Helper functions defined.\n\n")

Helper functions defined.



##(R): Section 4A — Build Coldata: Stratum A Studies

**Purpose:** Fetch GEO metadata and build per-sample coldata TSV files for the five Stratum A mouse brain studies: GSE149805, GSE198063, GSE144738, GSE189527, GSE201249. Each coldata file maps SRR accessions to condition labels (prion/control), sex, age, brain region, and other covariates extracted from GEO SOFT files.

**Inputs:** GEO SOFT files (downloaded/cached), `sample_meta` from Section 2.

**Outputs:** `../config/coldata/coldata_GSE*.tsv` for each Stratum A study.

**Key decisions:** `GSEMatrix=FALSE` is required to access per-sample `characteristics_ch1` fields — the series matrix format omits these. Each study has a bespoke parsing block because GEO metadata formats are not standardised.

In [9]:
%%R

# =============================================================================
# SECTION 4: PER-STUDY COLDATA CONSTRUCTION
# =============================================================================
# Each study has a dedicated block that:
#   1. Fetches GEO metadata
#   2. Resolves SRX -> SRR
#   3. Parses study-specific characteristics into standardised columns
#   4. Applies any study-specific sample filters
#   5. Writes the coldata CSV
# =============================================================================

all_coldata <- list()  # stores all coldata data.frames for final summary


In [12]:
%%R
# =============================================================================
# 4.1  GSE149805 — RML scrapie, N=176 valid samples (Stratum A)
#      Total downloaded: 186 SRRs. Valid for analysis: 176 (10 noCre excluded).
#
#      Characteristics: cre driver (=cell_type), time point, treatment (=condition)
#      condition: "non-infectious brain homogenate" -> control
#                 "RML prions" / "prion" -> prion
#      cell_type: CamKIIa, Cx3Cr1, GFAP, PV  (from "cre driver" field)
#
#      EXCLUDED (noCre, n=10): GSM4513119–4513128 / SRR11684370–379
#        These are "noCre" negative control samples — mice without a Cre
#        recombinase transgene. GEO metadata: cre driver="-", time point="-",
#        treatment="-". They have no cell-type specificity and no condition
#        assignment. They are technical controls for the RiboTag pulldown and
#        cannot be used in DESeq2. Exclusion is scientifically correct.
#        (The condition=NA filter in this script correctly removes them.)
# =============================================================================
cat("\n=== GSE149805 ===\n")

gse149805 <- fetch_geo("GSE149805")
gsm_srr_149805 <- build_gsm_srr_map(gse149805, "GSE149805")

coldata_149805 <- lapply(names(gse149805$gsms), function(gsm_id) {
  gsm   <- gse149805$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_149805$srr_id[gsm_srr_149805$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  # Map treatment -> condition
  treatment <- str_to_lower(chars[["treatment"]])
  condition <- case_when(
    str_detect(treatment, "non-infectious|nbh|control|mock|sham") ~ "control",
    str_detect(treatment, "prion|rml|scrapie|infectious")         ~ "prion",
    TRUE ~ NA_character_
  )

  # Map cre driver -> cell_type
  cell_type <- chars[["cre driver"]]

  # time_point: clean to numeric weeks (e.g., "16wpi" -> "16wpi")
  time_point <- chars[["time point"]]

  data.frame(
    sample_id  = srr,
    files      = quant_path(srr, "GSE149805"),
    condition  = condition,
    cell_type  = cell_type,
    time_point = time_point,
    sex        = str_to_lower(chars[["gender"]]),
    gsm_id     = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

write_coldata(coldata_149805, "GSE149805")
all_coldata[["GSE149805"]] <- coldata_149805
rm(gse149805, gsm_srr_149805); gc()


=== GSE149805 ===
  Fetching GEO metadata for GSE149805 (SOFT file)...
  GSE149805: 186 GSM samples found.
  [GSE149805] Resolving GSM -> SRX -> SRR for 186 samples...
  [GSE149805] Resolving 186 SRX accessions via NCBI Entrez...
  [GSE149805] Resolved 186/186 SRX->SRR mappings.
  [GSE149805] Written: ../config/coldata/GSE149805_coldata.csv (176 samples)
  [GSE149805] Condition counts: control=88, prion=88
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6522674 348.4    9928347 530.3  9928347 530.3
Vcells 11801493  90.1   21752374 166.0 17238289 131.6


Using locally cached version of GSE149805 found here:
../data/raw/geo_soft/GSE149805.soft.gz 
Reading file....
Parsing....
Found 187 entities...
GPL17021 (1 of 188 entities)
GSM4512979 (2 of 188 entities)
GSM4512980 (3 of 188 entities)
GSM4512981 (4 of 188 entities)
GSM4512982 (5 of 188 entities)
GSM4512983 (6 of 188 entities)
GSM4512984 (7 of 188 entities)
GSM4512985 (8 of 188 entities)
GSM4512986 (9 of 188 entities)
GSM4512987 (10 of 188 entities)
GSM4512988 (11 of 188 entities)
GSM4512989 (12 of 188 entities)
GSM4512990 (13 of 188 entities)
GSM4512991 (14 of 188 entities)
GSM4512992 (15 of 188 entities)
GSM4512993 (16 of 188 entities)
GSM4512994 (17 of 188 entities)
GSM4512995 (18 of 188 entities)
GSM4512996 (19 of 188 entities)
GSM4512997 (20 of 188 entities)
GSM4512998 (21 of 188 entities)
GSM4512999 (22 of 188 entities)
GSM4513000 (23 of 188 entities)
GSM4513001 (24 of 188 entities)
GSM4513002 (25 of 188 entities)
GSM4513003 (26 of 188 entities)
GSM4513004 (27 of 188 entities)
GS

In [13]:
%%R

# =============================================================================
# 4.2  GSE198063 — FFI D178N knock-in, N=58 valid samples (Stratum A)
#      Total downloaded: 159 SRRs. Valid for analysis: 58 (101 RiboTag IP excluded).
#
#      CRITICAL: This is a RiboTag study with two fractions per sample:
#        protocol="Input"   -> total RNA (n=58)  — USE THESE
#        protocol="RiboTag" -> ribosome-associated mRNA IP (n=101) — EXCLUDE
#
#      WHY EXCLUDE RiboTag IP: The IP fraction measures ribosome-associated mRNA
#      (translating mRNAs), a fundamentally different quantity from total RNA.
#      Including it would introduce a systematic technical bias incompatible with
#      the other 10 studies in this meta-analysis (all total RNA). The Input
#      fraction is standard bulk RNA-seq and is directly comparable.
#
#      condition: genotype="FFI" or "CJD" -> prion; genotype="Control" -> control
#      NOTE: Both FFI (Fatal Familial Insomnia) and CJD are prion diseases.
#            Combining them as "prion" is appropriate for the MCPS meta-analysis.
# =============================================================================
cat("\n=== GSE198063 ===\n")

gse198063 <- fetch_geo("GSE198063")
gsm_srr_198063 <- build_gsm_srr_map(gse198063, "GSE198063")

coldata_198063 <- lapply(names(gse198063$gsms), function(gsm_id) {
  gsm   <- gse198063$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_198063$srr_id[gsm_srr_198063$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  # FILTER: Input (total RNA) only — exclude RiboTag IP fraction
  protocol <- str_to_lower(chars[["protocol"]])
  if (!str_detect(protocol, "input")) return(NULL)

  # Map genotype -> condition
  genotype  <- str_to_lower(chars[["genotype"]])
  condition <- case_when(
    str_detect(genotype, "ffi|cjd|prion|d178n") ~ "prion",
    str_detect(genotype, "control|wt|wild")      ~ "control",
    TRUE ~ NA_character_
  )

  cell_type <- chars[["cell type"]]

  data.frame(
    sample_id  = srr,
    files      = quant_path(srr, "GSE198063"),
    condition  = condition,
    cell_type  = cell_type,
    gsm_id     = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

cat(sprintf("  [GSE198063] Input-only filter: %d samples retained (RiboTag IP excluded).\n",
            nrow(coldata_198063)))
write_coldata(coldata_198063, "GSE198063")
all_coldata[["GSE198063"]] <- coldata_198063
rm(gse198063, gsm_srr_198063); gc()


=== GSE198063 ===
  Fetching GEO metadata for GSE198063 (SOFT file)...
  GSE198063: 159 GSM samples found.
  [GSE198063] Resolving GSM -> SRX -> SRR for 159 samples...
  [GSE198063] Resolving 159 SRX accessions via NCBI Entrez...
  [GSE198063] Resolved 159/159 SRX->SRR mappings.
  [GSE198063] Input-only filter: 58 samples retained (RiboTag IP excluded).
  [GSE198063] Written: ../config/coldata/GSE198063_coldata.csv (58 samples)
  [GSE198063] Condition counts: control=18, prion=40
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6522888 348.4    9928347 530.3  9928347 530.3
Vcells 11802046  90.1   21752374 166.0 17238289 131.6


Using locally cached version of GSE198063 found here:
../data/raw/geo_soft/GSE198063.soft.gz 
Reading file....
Parsing....
Found 160 entities...
GPL24247 (1 of 161 entities)
GSM5937046 (2 of 161 entities)
GSM5937047 (3 of 161 entities)
GSM5937048 (4 of 161 entities)
GSM5937049 (5 of 161 entities)
GSM5937050 (6 of 161 entities)
GSM5937051 (7 of 161 entities)
GSM5937052 (8 of 161 entities)
GSM5937053 (9 of 161 entities)
GSM5937054 (10 of 161 entities)
GSM5937055 (11 of 161 entities)
GSM5937056 (12 of 161 entities)
GSM5937057 (13 of 161 entities)
GSM5937058 (14 of 161 entities)
GSM5937059 (15 of 161 entities)
GSM5937060 (16 of 161 entities)
GSM5937061 (17 of 161 entities)
GSM5937062 (18 of 161 entities)
GSM5937063 (19 of 161 entities)
GSM5937064 (20 of 161 entities)
GSM5937065 (21 of 161 entities)
GSM5937066 (22 of 161 entities)
GSM5937067 (23 of 161 entities)
GSM5937068 (24 of 161 entities)
GSM5937069 (25 of 161 entities)
GSM5937070 (26 of 161 entities)
GSM5937071 (27 of 161 entities)
GS

In [17]:
%%R

# =============================================================================
# 4.3  GSE144738 — RML scrapie, N=137, pre-clinical time course (Stratum A)
#      Characteristics: treatment (=condition), date of sacrifice (=time_point),
#                       genotype (PrP present = WT), tissue
#      condition: "non-infectious brain homogenate" -> control
#                 "RML6 prions" -> prion
#      time_point: "date of sacrifice" field (e.g., "4 wpi", "8 wpi", ...)
#      layout: MIXED — 131 PAIRED + 6 SINGLE (SRR11999701–SRR11999706).
#              Layout is joined from sample_metadata.tsv and added as a
#              covariate column so DESeq2 can control for this technical
#              batch effect (design: ~ time_point + layout + condition).
#              All other studies are layout-homogeneous so layout is
#              confounded with study identity and cannot be a covariate there.
# =============================================================================
cat("\n=== GSE144738 ===\n")
gse144738 <- fetch_geo("GSE144738")
gsm_srr_144738 <- build_gsm_srr_map(gse144738, "GSE144738")

coldata_144738 <- lapply(names(gse144738$gsms), function(gsm_id) {
  gsm   <- gse144738$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_144738$srr_id[gsm_srr_144738$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  treatment <- str_to_lower(chars[["treatment"]])
  condition <- case_when(
    str_detect(treatment, "non-infectious|nbh|control|mock") ~ "control",
    str_detect(treatment, "prion|rml|scrapie|infectious")    ~ "prion",
    TRUE ~ NA_character_
  )

  data.frame(
    sample_id  = srr,
    files      = quant_path(srr, "GSE144738"),
    condition  = condition,
    time_point = chars[["date of sacrifice"]],
    tissue     = chars[["tissue"]],
    gsm_id     = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

# Add layout column from sample_metadata.tsv (ORIGINAL SCRIPT BEHAVIOR)
# GSE144738 is the only mixed-layout study (131 PAIRED + 6 SINGLE).
coldata_144738 <- coldata_144738 %>%
  left_join(
    sample_meta %>%
      filter(GSE == "GSE144738") %>%
      transmute(sample_id = SRR,
                layout    = str_to_lower(LAYOUT)),
    by = "sample_id"
  )

n_paired <- sum(coldata_144738$layout == "paired", na.rm = TRUE)
n_single <- sum(coldata_144738$layout == "single", na.rm = TRUE)
n_na     <- sum(is.na(coldata_144738$layout))
cat(sprintf("  [GSE144738] Layout: %d paired, %d single, %d NA\n", n_paired, n_single, n_na))

write_coldata(coldata_144738, "GSE144738")
all_coldata[["GSE144738"]] <- coldata_144738
rm(gse144738, gsm_srr_144738); gc()


=== GSE144738 ===
  Fetching GEO metadata for GSE144738 (SOFT file)...
  GSE144738: 137 GSM samples found.
  [GSE144738] Resolving GSM -> SRX -> SRR for 137 samples...
  [GSE144738] Resolving 137 SRX accessions via NCBI Entrez...
  [GSE144738] Resolved 137/137 SRX->SRR mappings.
  [GSE144738] Layout: 131 paired, 6 single, 0 NA
  [GSE144738] Written: ../config/coldata/GSE144738_coldata.csv (137 samples)
  [GSE144738] Condition counts: control=66, prion=71
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6523263 348.4    9928347 530.3  9928347 530.3
Vcells 11802549  90.1   21752374 166.0 17238289 131.6


Using locally cached version of GSE144738 found here:
../data/raw/geo_soft/GSE144738.soft.gz 
Reading file....
Parsing....
Found 139 entities...
GPL21103 (1 of 140 entities)
GPL24247 (2 of 140 entities)
GSM4294560 (3 of 140 entities)
GSM4294561 (4 of 140 entities)
GSM4294562 (5 of 140 entities)
GSM4294563 (6 of 140 entities)
GSM4294564 (7 of 140 entities)
GSM4294565 (8 of 140 entities)
GSM4294566 (9 of 140 entities)
GSM4294567 (10 of 140 entities)
GSM4294568 (11 of 140 entities)
GSM4294569 (12 of 140 entities)
GSM4294570 (13 of 140 entities)
GSM4294571 (14 of 140 entities)
GSM4294572 (15 of 140 entities)
GSM4294573 (16 of 140 entities)
GSM4294574 (17 of 140 entities)
GSM4294575 (18 of 140 entities)
GSM4294576 (19 of 140 entities)
GSM4294577 (20 of 140 entities)
GSM4294578 (21 of 140 entities)
GSM4294579 (22 of 140 entities)
GSM4294580 (23 of 140 entities)
GSM4294581 (24 of 140 entities)
GSM4294582 (25 of 140 entities)
GSM4294583 (26 of 140 entities)
GSM4294584 (27 of 140 entities)
GSM4

In [18]:
%%R

# =============================================================================
# 4.4  GSE189527 — RML scrapie, N=98, pre-clinical, 6 cell types (Stratum A)
#      Characteristics: condition (=condition), timepoint (=time_point),
#                       cell type (=cell_type)
#      condition: "Control" -> control; "RML" -> prion
# =============================================================================
cat("\n=== GSE189527 ===\n")

gse189527 <- fetch_geo("GSE189527")
gsm_srr_189527 <- build_gsm_srr_map(gse189527, "GSE189527")

coldata_189527 <- lapply(names(gse189527$gsms), function(gsm_id) {
  gsm   <- gse189527$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_189527$srr_id[gsm_srr_189527$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  raw_condition <- str_to_lower(chars[["condition"]])
  condition <- case_when(
    str_detect(raw_condition, "control|mock|nbh|sham") ~ "control",
    str_detect(raw_condition, "rml|prion|scrapie")     ~ "prion",
    TRUE ~ NA_character_
  )

  data.frame(
    sample_id  = srr,
    files      = quant_path(srr, "GSE189527"),
    condition  = condition,
    time_point = chars[["timepoint"]],
    cell_type  = chars[["cell type"]],
    gsm_id     = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

write_coldata(coldata_189527, "GSE189527")
all_coldata[["GSE189527"]] <- coldata_189527
rm(gse189527, gsm_srr_189527); gc()


=== GSE189527 ===
  Fetching GEO metadata for GSE189527 (SOFT file)...
  GSE189527: 98 GSM samples found.
  [GSE189527] Resolving GSM -> SRX -> SRR for 98 samples...
  [GSE189527] Resolving 98 SRX accessions via NCBI Entrez...
  [GSE189527] Resolved 98/98 SRX->SRR mappings.
  [GSE189527] Written: ../config/coldata/GSE189527_coldata.csv (98 samples)
  [GSE189527] Condition counts: control=50, prion=48
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6523617 348.4    9928347 530.3  9928347 530.3
Vcells 11803863  90.1   21752374 166.0 17238289 131.6


Using locally cached version of GSE189527 found here:
../data/raw/geo_soft/GSE189527.soft.gz 
Reading file....
Parsing....
Found 99 entities...
GPL17021 (1 of 100 entities)
GSM5704762 (2 of 100 entities)
GSM5704763 (3 of 100 entities)
GSM5704764 (4 of 100 entities)
GSM5704765 (5 of 100 entities)
GSM5704766 (6 of 100 entities)
GSM5704767 (7 of 100 entities)
GSM5704768 (8 of 100 entities)
GSM5704769 (9 of 100 entities)
GSM5704770 (10 of 100 entities)
GSM5704771 (11 of 100 entities)
GSM5704772 (12 of 100 entities)
GSM5704773 (13 of 100 entities)
GSM5704774 (14 of 100 entities)
GSM5704775 (15 of 100 entities)
GSM5704776 (16 of 100 entities)
GSM5704777 (17 of 100 entities)
GSM5704778 (18 of 100 entities)
GSM5704779 (19 of 100 entities)
GSM5704780 (20 of 100 entities)
GSM5704781 (21 of 100 entities)
GSM5704782 (22 of 100 entities)
GSM5704783 (23 of 100 entities)
GSM5704784 (24 of 100 entities)
GSM5704785 (25 of 100 entities)
GSM5704786 (26 of 100 entities)
GSM5704787 (27 of 100 entities)
GSM

In [19]:
%%R

# =============================================================================
# 4.5  GSE201249 — RML scrapie, N=50, terminal, CA1 + Thalamus (Stratum A)
#      Characteristics: tissue (=brain_region), treatment (=condition),
#                       genotype (all CD1 = WT)
#      condition: "Mock" -> control; "RML" -> prion
#      brain_region: "CA1" -> CA1; "thalamus" -> thalamus
# =============================================================================
cat("\n=== GSE201249 ===\n")

gse201249 <- fetch_geo("GSE201249")
gsm_srr_201249 <- build_gsm_srr_map(gse201249, "GSE201249")

coldata_201249 <- lapply(names(gse201249$gsms), function(gsm_id) {
  gsm   <- gse201249$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_201249$srr_id[gsm_srr_201249$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  treatment <- str_to_lower(chars[["treatment"]])
  condition <- case_when(
    str_detect(treatment, "mock|control|nbh|sham") ~ "control",
    str_detect(treatment, "rml|prion|scrapie")     ~ "prion",
    TRUE ~ NA_character_
  )

  # Standardise brain_region to lowercase
  brain_region <- str_to_lower(str_trim(chars[["tissue"]]))

  data.frame(
    sample_id    = srr,
    files        = quant_path(srr, "GSE201249"),
    condition    = condition,
    brain_region = brain_region,
    gsm_id       = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

# NOTE: GSE201249 has no cell_type field in GEO metadata.
# The design ~ brain_region + cell_type + condition in 05_ will fall back
# to ~ brain_region + condition (the rank-deficiency guard handles this).
cat("  [GSE201249] Note: no cell_type field in GEO metadata. Design will use ~ brain_region + condition.\n")

write_coldata(coldata_201249, "GSE201249")
all_coldata[["GSE201249"]] <- coldata_201249
rm(gse201249, gsm_srr_201249); gc()


=== GSE201249 ===
  Fetching GEO metadata for GSE201249 (SOFT file)...
  GSE201249: 50 GSM samples found.
  [GSE201249] Resolving GSM -> SRX -> SRR for 50 samples...
  [GSE201249] Resolving 50 SRX accessions via NCBI Entrez...
  [GSE201249] Resolved 50/50 SRX->SRR mappings.
  [GSE201249] Note: no cell_type field in GEO metadata. Design will use ~ brain_region + condition.
  [GSE201249] Written: ../config/coldata/GSE201249_coldata.csv (50 samples)
  [GSE201249] Condition counts: control=25, prion=25
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6523697 348.5    9928347 530.3  9928347 530.3
Vcells 11804357  90.1   21752374 166.0 17238289 131.6


Using locally cached version of GSE201249 found here:
../data/raw/geo_soft/GSE201249.soft.gz 
Reading file....
Parsing....
Found 51 entities...
GPL21626 (1 of 52 entities)
GSM6056549 (2 of 52 entities)
GSM6056550 (3 of 52 entities)
GSM6056551 (4 of 52 entities)
GSM6056552 (5 of 52 entities)
GSM6056553 (6 of 52 entities)
GSM6056554 (7 of 52 entities)
GSM6056555 (8 of 52 entities)
GSM6056556 (9 of 52 entities)
GSM6056557 (10 of 52 entities)
GSM6056558 (11 of 52 entities)
GSM6056559 (12 of 52 entities)
GSM6056560 (13 of 52 entities)
GSM6056561 (14 of 52 entities)
GSM6056562 (15 of 52 entities)
GSM6056563 (16 of 52 entities)
GSM6056564 (17 of 52 entities)
GSM6056565 (18 of 52 entities)
GSM6056566 (19 of 52 entities)
GSM6056567 (20 of 52 entities)
GSM6056568 (21 of 52 entities)
GSM6056569 (22 of 52 entities)
GSM6056570 (23 of 52 entities)
GSM6056571 (24 of 52 entities)
GSM6056572 (25 of 52 entities)
GSM6056573 (26 of 52 entities)
GSM6056574 (27 of 52 entities)
GSM6056575 (28 of 52 entities)

In [20]:
%%R

# =============================================================================
# 4.6  GSE210128 — Peripheral blood/spleen/muscle, N=173 (Stratum B)
#      Characteristics: genotype (C57BL/6), tissue (=tissue), treatment (=condition)
#      condition: "NBH" -> control; "RML6" -> prion
#      tissue: "peripheral, whole blood", "spleen", "skeletal muscle"
#      time_point: parsed from GSM title (e.g. "4 wpi", "term wpi" -> 22)
#        Timepoints: 4, 8, 12, 14, 16, 18, 20 wpi + terminal
#        Terminal is encoded as 22 wpi (RML6 in C57BL/6 reaches terminal ~22 wpi;
#        exact value is not critical — it is used only as an ordered numeric for
#        LRT trajectory modelling and is clearly labelled in all outputs).
#      cohort: "main cohort" or "validation cohort" (from GSM title)
#      NOTE: Used by Stream 3 (peripheral analysis) only.
#      Source: GEO GSE210128 (Aguzzi et al., University Hospital Zurich, 2022)
# =============================================================================
cat("\n=== GSE210128 ===\n")

gse210128 <- fetch_geo("GSE210128")
gsm_srr_210128 <- build_gsm_srr_map(gse210128, "GSE210128")

coldata_210128 <- lapply(names(gse210128$gsms), function(gsm_id) {
  gsm        <- gse210128$gsms[[gsm_id]]
  chars      <- parse_characteristics(gsm)
  gsm_title  <- str_to_lower(str_trim(GEOquery::Meta(gsm)$title[[1]]))
  srr        <- gsm_srr_210128$srr_id[gsm_srr_210128$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  # --- condition ---
  treatment <- str_to_lower(chars[["treatment"]])
  condition <- case_when(
    str_detect(treatment, "nbh|control|mock|sham") ~ "control",
    str_detect(treatment, "rml|prion|scrapie")     ~ "prion",
    TRUE ~ NA_character_
  )

  # --- tissue ---
  tissue_raw <- str_to_lower(str_trim(chars[["tissue"]]))
  tissue <- case_when(
    str_detect(tissue_raw, "blood")   ~ "whole_blood",
    str_detect(tissue_raw, "spleen")  ~ "spleen",
    str_detect(tissue_raw, "muscle")  ~ "skeletal_muscle",
    TRUE ~ tissue_raw
  )

  # --- time_point (numeric wpi) ---
  # GSM titles follow the pattern: "main cohort, blood, RML6, 16 wpi, rep 2"
  # "term wpi" is encoded as 22 (approximate terminal endpoint for RML6/C57BL6).
  tp_raw <- str_extract(gsm_title, "(\\d+|term)\\s*wpi")
  time_point_wpi <- if (!is.na(tp_raw)) {
    tp_num <- str_extract(tp_raw, "\\d+|term")
    if (tp_num == "term") 22L else as.integer(tp_num)
  } else {
    NA_integer_
  }

  # --- cohort ---
  cohort <- case_when(
    str_detect(gsm_title, "validation cohort") ~ "validation",
    str_detect(gsm_title, "main cohort")       ~ "main",
    TRUE ~ NA_character_
  )

  data.frame(
    sample_id      = srr,
    files          = quant_path(srr, "GSE210128"),
    condition      = condition,
    tissue         = tissue,
    time_point_wpi = time_point_wpi,
    cohort         = cohort,
    gsm_id         = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

# Sanity check: report timepoint × tissue × condition counts
cat("  GSE210128 sample breakdown:\n")
print(with(coldata_210128,
  table(tissue, time_point_wpi, condition)))

write_coldata(coldata_210128, "GSE210128")
all_coldata[["GSE210128"]] <- coldata_210128
rm(gse210128, gsm_srr_210128); gc()


=== GSE210128 ===
  Fetching GEO metadata for GSE210128 (SOFT file)...
  GSE210128: 173 GSM samples found.
  [GSE210128] Resolving GSM -> SRX -> SRR for 173 samples...
  [GSE210128] Resolving 173 SRX accessions via NCBI Entrez...
  [GSE210128] Resolved 173/173 SRX->SRR mappings.
  GSE210128 sample breakdown:
, , condition = control

                 time_point_wpi
tissue            4 8 12 14 16 18 20 22
  skeletal_muscle 3 6  6  3  6  3  6  6
  spleen          3 3  3  3  3  3  3  3
  whole_blood     3 3  2  3  3  3  3  3

, , condition = prion

                 time_point_wpi
tissue            4 8 12 14 16 18 20 22
  skeletal_muscle 3 6  6  3  6  3  6  5
  spleen          3 3  3  3  3  3  3  3
  whole_blood     3 3  4  3  3  3  3  3

  [GSE210128] Written: ../config/coldata/GSE210128_coldata.csv (173 samples)
  [GSE210128] Condition counts: control=86, prion=87
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6523509 348.4    9928347 530.3  9928347 530.3
Vcells 11803032  

Using locally cached version of GSE210128 found here:
../data/raw/geo_soft/GSE210128.soft.gz 
Reading file....
Parsing....
Found 174 entities...
GPL21103 (1 of 175 entities)
GSM6421805 (2 of 175 entities)
GSM6421806 (3 of 175 entities)
GSM6421807 (4 of 175 entities)
GSM6421808 (5 of 175 entities)
GSM6421809 (6 of 175 entities)
GSM6421810 (7 of 175 entities)
GSM6421811 (8 of 175 entities)
GSM6421812 (9 of 175 entities)
GSM6421813 (10 of 175 entities)
GSM6421817 (11 of 175 entities)
GSM6421818 (12 of 175 entities)
GSM6421819 (13 of 175 entities)
GSM6421823 (14 of 175 entities)
GSM6421824 (15 of 175 entities)
GSM6421828 (16 of 175 entities)
GSM6421829 (17 of 175 entities)
GSM6421830 (18 of 175 entities)
GSM6421831 (19 of 175 entities)
GSM6421834 (20 of 175 entities)
GSM6421835 (21 of 175 entities)
GSM6421836 (22 of 175 entities)
GSM6421837 (23 of 175 entities)
GSM6421838 (24 of 175 entities)
GSM6421839 (25 of 175 entities)
GSM6421840 (26 of 175 entities)
GSM6421841 (27 of 175 entities)
GS

In [21]:
%%R

# =============================================================================
# 4.7  GSE214373 — Human sCJD, N=14, Frontal Cortex (Stratum C)
#      Characteristics: condition, sex, age (continuous), tissue
#      condition: "sCJD" -> prion; "control" -> control
#      NOTE: Used by Stream 2 (human validation) only.
#
#      Age source: Individual ages are NOT in GEO metadata. They were retrieved
#      from Supplementary Table 2 ("Demographics" sheet) of the source preprint:
#        Dimitriadis et al. (2022) bioRxiv 2022.09.13.507650
#        https://www.biorxiv.org/content/10.1101/2022.09.13.507650v1
#      The PDG codes in GEO sample titles (e.g. "53846_sCJD") were used to
#      cross-reference individual rows in Supp Table 2.
#
#      Age confound assessment (from Supp Table 2, N=14):
#        sCJD (N=5):    mean 71.6y ± 4.2 SD  (range 67–77)
#        Control (N=9): mean 83.3y ± 9.6 SD  (range 69–103)
#        Welch t-test: t=-2.57, p=0.024; Mann-Whitney U: p=0.012
#        Age-condition Pearson r=-0.60 (p=0.024); VIF(condition)=2.06
#      Decision: include age as continuous covariate (~ age + sex + condition).
#      Design matrix is full rank (rank=4/4, condition number=946).
#      Caveat: age and condition are correlated (r=-0.60); age coefficient
#      will be imprecisely estimated but condition estimate is still valid.
#      Sex imbalance (sCJD 4F/1M vs control 5M/4F) is not significant
#      (Fisher's exact p=0.30); sex included as covariate regardless.
# =============================================================================
cat("\n=== GSE214373 ===\n")

# Hard-coded individual ages from Supp Table 2 (Dimitriadis et al. 2022 bioRxiv).
# Key: GSM accession -> PDG code (from GEO sample title) -> age from Supp Table 2.
GSE214373_age_map <- c(
  GSM6604800 = 74L,   # PDG 53846, sCJD, F
  GSM6604801 = 67L,   # PDG 54580, sCJD, M
  GSM6604802 = 68L,   # PDG 56720, sCJD, F
  GSM6604803 = 77L,   # PDG 54466, sCJD, F
  GSM6604804 = 72L,   # PDG 56093, sCJD, F
  GSM6604805 = 81L,   # PDG 67094, control, M
  GSM6604806 = 69L,   # PDG 67096, control, M
  GSM6604807 = 87L,   # PDG 67097, control, M
  GSM6604808 = 76L,   # PDG 67098, control, M
  GSM6604809 = 89L,   # PDG 67099, control, M
  GSM6604810 = 79L,   # PDG 67104, control, F
  GSM6604811 = 80L,   # PDG 67105, control, F
  GSM6604812 = 86L,   # PDG 67106, control, F
  GSM6604813 = 103L   # PDG 67109, control, F
)

gse214373 <- fetch_geo("GSE214373")
gsm_srr_214373 <- build_gsm_srr_map(gse214373, "GSE214373")

coldata_214373 <- lapply(names(gse214373$gsms), function(gsm_id) {
  gsm   <- gse214373$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_214373$srr_id[gsm_srr_214373$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  # Condition: derive from GEO sample title (e.g. "53846_sCJD" or "67094_control")
  sample_title  <- GEOquery::Meta(gsm)$title[[1]]
  raw_condition <- str_to_lower(sample_title)
  condition <- case_when(
    str_detect(raw_condition, "scjd|cjd|prion") ~ "prion",
    str_detect(raw_condition, "control|normal")  ~ "control",
    TRUE ~ NA_character_
  )

  # Age: from hard-coded map (Supp Table 2, Dimitriadis et al. 2022 bioRxiv)
  age <- GSE214373_age_map[gsm_id]
  if (is.na(age)) {
    cat(sprintf("  [GSE214373] WARNING: no age found for %s — check GSE214373_age_map\n", gsm_id))
  }

  data.frame(
    sample_id = srr,
    files     = quant_path(srr, "GSE214373"),
    condition = condition,
    sex       = str_to_lower(str_trim(chars[["sex"]])),
    age       = as.integer(age),
    tissue    = "frontal_cortex",
    gsm_id    = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

# Sanity check: verify age coverage
n_missing_age <- sum(is.na(coldata_214373$age))
if (n_missing_age > 0) {
  stop(sprintf("[GSE214373] %d samples have missing age — check GSE214373_age_map", n_missing_age))
}
cat(sprintf("  [GSE214373] N=%d samples; age range %d–%d\n",
            nrow(coldata_214373),
            min(coldata_214373$age, na.rm = TRUE),
            max(coldata_214373$age, na.rm = TRUE)))
cat(sprintf("  [GSE214373] sCJD mean age: %.1f  Control mean age: %.1f\n",
            mean(coldata_214373$age[coldata_214373$condition == "prion"]),
            mean(coldata_214373$age[coldata_214373$condition == "control"])))
cat("  [GSE214373] DESeq2 design will be: ~ age + sex + condition\n")
cat("  [GSE214373] NOTE: age-condition r=-0.60 (p=0.024); condition estimate\n")
cat("              is valid but interpret with caution given small N and age imbalance.\n")

write_coldata(coldata_214373, "GSE214373")
all_coldata[["GSE214373"]] <- coldata_214373
rm(gse214373, gsm_srr_214373); gc()


=== GSE214373 ===
  Fetching GEO metadata for GSE214373 (SOFT file)...
  GSE214373: 14 GSM samples found.
  [GSE214373] Resolving GSM -> SRX -> SRR for 14 samples...
  [GSE214373] Resolving 14 SRX accessions via NCBI Entrez...
  [GSE214373] Resolved 14/14 SRX->SRR mappings.
  [GSE214373] N=14 samples; age range 67–103
  [GSE214373] sCJD mean age: 71.6  Control mean age: 83.3
  [GSE214373] DESeq2 design will be: ~ age + sex + condition
  [GSE214373] NOTE: age-condition r=-0.60 (p=0.024); condition estimate
              is valid but interpret with caution given small N and age imbalance.
  [GSE214373] Written: ../config/coldata/GSE214373_coldata.csv (14 samples)
  [GSE214373] Condition counts: control=9, prion=5
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6524312 348.5    9928347 530.3  9928347 530.3
Vcells 11805742  90.1   21752374 166.0 17239063 131.6


Using locally cached version of GSE214373 found here:
../data/raw/geo_soft/GSE214373.soft.gz 
Reading file....
Parsing....
Found 15 entities...
GPL18573 (1 of 16 entities)
GSM6604800 (2 of 16 entities)
GSM6604801 (3 of 16 entities)
GSM6604802 (4 of 16 entities)
GSM6604803 (5 of 16 entities)
GSM6604804 (6 of 16 entities)
GSM6604805 (7 of 16 entities)
GSM6604806 (8 of 16 entities)
GSM6604807 (9 of 16 entities)
GSM6604808 (10 of 16 entities)
GSM6604809 (11 of 16 entities)
GSM6604810 (12 of 16 entities)
GSM6604811 (13 of 16 entities)
GSM6604812 (14 of 16 entities)
GSM6604813 (15 of 16 entities)


In [22]:
%%R

# =============================================================================
# 4.8  GSE90977 — sCJD mouse model, N=16, 2 time points (Stratum D)
#      Characteristics: condition (=condition), time point (=time_point)
#      condition: "Control challenged mouse" -> control
#                 "sporadic Creutzfeldt-Jakob MM1 challenged mouse" -> prion
# =============================================================================
cat("\n=== GSE90977 ===\n")

gse90977 <- fetch_geo("GSE90977")
gsm_srr_90977 <- build_gsm_srr_map(gse90977, "GSE90977")

coldata_90977 <- lapply(names(gse90977$gsms), function(gsm_id) {
  gsm   <- gse90977$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_90977$srr_id[gsm_srr_90977$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  raw_condition <- str_to_lower(chars[["condition"]])
  condition <- case_when(
    str_detect(raw_condition, "control")                          ~ "control",
    str_detect(raw_condition, "creutzfeldt|jakob|cjd|prion|scjd") ~ "prion",
    TRUE ~ NA_character_
  )

  data.frame(
    sample_id  = srr,
    files      = quant_path(srr, "GSE90977"),
    condition  = condition,
    time_point = chars[["time point"]],
    gsm_id     = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

write_coldata(coldata_90977, "GSE90977")
all_coldata[["GSE90977"]] <- coldata_90977
rm(gse90977, gsm_srr_90977); gc()


=== GSE90977 ===
  Fetching GEO metadata for GSE90977 (SOFT file)...
  GSE90977: 16 GSM samples found.
  [GSE90977] Resolving GSM -> SRX -> SRR for 16 samples...
  [GSE90977] Resolving 16 SRX accessions via NCBI Entrez...
  [GSE90977] Resolved 16/16 SRX->SRR mappings.
  [GSE90977] Written: ../config/coldata/GSE90977_coldata.csv (16 samples)
  [GSE90977] Condition counts: control=8, prion=8
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6524416 348.5    9928347 530.3  9928347 530.3
Vcells 11806351  90.1   21752374 166.0 17239063 131.6


Reading file....
Parsing....
Found 17 entities...
GPL13112 (1 of 18 entities)
GSM2418613 (2 of 18 entities)
GSM2418614 (3 of 18 entities)
GSM2418615 (4 of 18 entities)
GSM2418616 (5 of 18 entities)
GSM2418617 (6 of 18 entities)
GSM2418618 (7 of 18 entities)
GSM2418619 (8 of 18 entities)
GSM2418620 (9 of 18 entities)
GSM2418621 (10 of 18 entities)
GSM2418622 (11 of 18 entities)
GSM2418623 (12 of 18 entities)
GSM2418624 (13 of 18 entities)
GSM2418625 (14 of 18 entities)
GSM2418626 (15 of 18 entities)
GSM2418627 (16 of 18 entities)
GSM2418628 (17 of 18 entities)


In [23]:
%%R

# =============================================================================
# 4.9  GSE124932 — SARM1 KO prion model, N=8 valid samples (Stratum D)
#      Total downloaded: 16 SRRs. Valid for analysis: 8 (8 SARM1-/- KO excluded).
#
#      Characteristics: genotype, infected with (=condition), tissue, age
#      FILTER: WT arms only (genotype="Wild type"). SARM1-/- excluded (n=8).
#
#      WHY EXCLUDE SARM1-/-: SARM1 (Sterile Alpha and TIR Motif Containing 1) is
#      a key regulator of axon degeneration (Wallerian degeneration). SARM1-/-
#      mice have dramatically altered neurodegeneration kinetics and transcriptional
#      responses. Including SARM1-/- samples would confound the prion disease
#      transcriptional signature with SARM1-dependent pathways. Only WT vs. WT
#      comparisons are valid for the MCPS meta-analysis.
#
#      condition: "not infected" -> control; "RML" / "infected" -> prion
# =============================================================================
cat("\n=== GSE124932 ===\n")

gse124932 <- fetch_geo("GSE124932")
gsm_srr_124932 <- build_gsm_srr_map(gse124932, "GSE124932")

coldata_124932 <- lapply(names(gse124932$gsms), function(gsm_id) {
  gsm   <- gse124932$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_124932$srr_id[gsm_srr_124932$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  # FILTER: Wild type only
  genotype <- str_to_lower(chars[["genotype"]])
  if (!str_detect(genotype, "wild type|wt|wildtype")) return(NULL)

  infected <- str_to_lower(chars[["infected with"]])
  condition <- case_when(
    str_detect(infected, "not infected|uninfected|control|mock|nbh") ~ "control",
    str_detect(infected, "rml|prion|scrapie|infected")               ~ "prion",
    TRUE ~ NA_character_
  )

  data.frame(
    sample_id = srr,
    files     = quant_path(srr, "GSE124932"),
    condition = condition,
    genotype  = chars[["genotype"]],
    age       = chars[["age"]],
    gsm_id    = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

cat(sprintf("  [GSE124932] WT-only filter: %d samples retained (KO arms excluded).\n",
            nrow(coldata_124932)))
write_coldata(coldata_124932, "GSE124932")
all_coldata[["GSE124932"]] <- coldata_124932
rm(gse124932, gsm_srr_124932); gc()


=== GSE124932 ===
  Fetching GEO metadata for GSE124932 (SOFT file)...
  GSE124932: 16 GSM samples found.
  [GSE124932] Resolving GSM -> SRX -> SRR for 16 samples...
  [GSE124932] Resolving 16 SRX accessions via NCBI Entrez...
  [GSE124932] Resolved 16/16 SRX->SRR mappings.
  [GSE124932] WT-only filter: 8 samples retained (KO arms excluded).
  [GSE124932] Written: ../config/coldata/GSE124932_coldata.csv (8 samples)
  [GSE124932] Condition counts: control=4, prion=4
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6524682 348.5    9928347 530.3  9928347 530.3
Vcells 11807000  90.1   21752374 166.0 17239063 131.6


Reading file....
Parsing....
Found 17 entities...
GPL21103 (1 of 18 entities)
GSM3559574 (2 of 18 entities)
GSM3559575 (3 of 18 entities)
GSM3559576 (4 of 18 entities)
GSM3559577 (5 of 18 entities)
GSM3559578 (6 of 18 entities)
GSM3559579 (7 of 18 entities)
GSM3559580 (8 of 18 entities)
GSM3559581 (9 of 18 entities)
GSM3559582 (10 of 18 entities)
GSM3559583 (11 of 18 entities)
GSM3559584 (12 of 18 entities)
GSM3559585 (13 of 18 entities)
GSM3559586 (14 of 18 entities)
GSM3559587 (15 of 18 entities)
GSM3559588 (16 of 18 entities)
GSM3559589 (17 of 18 entities)


In [26]:
%%R

# =============================================================================
# 4.10 GSE207251 — 22L prion, N=6, hippocampus, terminal (Stratum D)
#      Characteristics: treatment (=condition), genotype (all WT), sex, tissue
#      condition: "prion (22L stain) intracraneal inoculation" -> prion
#                 "mock" / "NBH" -> control
# =============================================================================
cat("\n=== GSE207251 ===\n")

gse207251 <- fetch_geo("GSE207251")
gsm_srr_207251 <- build_gsm_srr_map(gse207251, "GSE207251")

coldata_207251 <- lapply(names(gse207251$gsms), function(gsm_id) {
  gsm   <- gse207251$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_207251$srr_id[gsm_srr_207251$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  treatment <- str_to_lower(chars[["treatment"]])
  condition <- case_when(
    str_detect(treatment, "prion|rml|scrapie|22l|infectious") ~ "prion",
    str_detect(treatment, "mock|nbh|control|sham")            ~ "control",
    TRUE ~ NA_character_
  )

  data.frame(
    sample_id = srr,
    files     = quant_path(srr, "GSE207251"),
    condition = condition,
    sex       = str_to_lower(str_trim(chars[["sex"]])),
    gsm_id    = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

write_coldata(coldata_207251, "GSE207251")
all_coldata[["GSE207251"]] <- coldata_207251
rm(gse207251, gsm_srr_207251); gc()


=== GSE207251 ===
  Fetching GEO metadata for GSE207251 (SOFT file)...
  GSE207251: 6 GSM samples found.
  [GSE207251] Resolving GSM -> SRX -> SRR for 6 samples...
  [GSE207251] Resolving 6 SRX accessions via NCBI Entrez...
  [GSE207251] Resolved 6/6 SRX->SRR mappings.
  [GSE207251] Written: ../config/coldata/GSE207251_coldata.csv (6 samples)
  [GSE207251] Condition counts: control=3, prion=3
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6524895 348.5    9928347 530.3  9928347 530.3
Vcells 11807629  90.1   21752374 166.0 17239063 131.6


Reading file....
Parsing....
Found 7 entities...
GPL21103 (1 of 8 entities)
GSM6283031 (2 of 8 entities)
GSM6283032 (3 of 8 entities)
GSM6283033 (4 of 8 entities)
GSM6283034 (5 of 8 entities)
GSM6283035 (6 of 8 entities)
GSM6283036 (7 of 8 entities)


In [27]:
%%R

# =============================================================================
# 4.11 GSE202275 — Tg37 prion + VU846 drug, N=6 valid samples (Stratum D)
#      Total downloaded: 12 SRRs. Valid for analysis: 6 (6 VU846-treated excluded).
#
#      Characteristics: genotype (Hemizygous Tg37), inoculation, treatment
#      FILTER: vehicle arms only (treatment contains "vehicle"). VU846 excluded (n=6).
#
#      WHY EXCLUDE VU846-TREATED: VU846 is a mGluR5 negative allosteric modulator
#      being tested as a potential prion disease therapeutic. VU846-treated samples
#      represent a drug intervention that alters the transcriptional landscape.
#      Including them would conflate drug effects with natural prion disease
#      progression, contaminating the MCPS signature. Only vehicle-arm samples
#      (prion + vehicle vs. control + vehicle) represent unmodified prion disease.
#
#      condition: "prion + vehicle" -> prion; "control + vehicle" -> control
# =============================================================================
cat("\n=== GSE202275 ===\n")

gse202275 <- fetch_geo("GSE202275")
gsm_srr_202275 <- build_gsm_srr_map(gse202275, "GSE202275")

coldata_202275 <- lapply(names(gse202275$gsms), function(gsm_id) {
  gsm   <- gse202275$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_202275$srr_id[gsm_srr_202275$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  treatment <- str_to_lower(chars[["treatment"]])

  # FILTER: vehicle arms only (exclude VU846-treated arms)
  if (!str_detect(treatment, "vehicle")) return(NULL)

  condition <- case_when(
    str_detect(treatment, "prion")   ~ "prion",
    str_detect(treatment, "control") ~ "control",
    TRUE ~ NA_character_
  )

  data.frame(
    sample_id  = srr,
    files      = quant_path(srr, "GSE202275"),
    condition  = condition,
    inoculation = chars[["inoculation"]],
    gsm_id     = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

cat(sprintf("  [GSE202275] Vehicle-only filter: %d samples retained (VU846 arms excluded).\n",
            nrow(coldata_202275)))
write_coldata(coldata_202275, "GSE202275")
all_coldata[["GSE202275"]] <- coldata_202275
rm(gse202275, gsm_srr_202275); gc()


=== GSE202275 ===
  Fetching GEO metadata for GSE202275 (SOFT file)...
  GSE202275: 12 GSM samples found.
  [GSE202275] Resolving GSM -> SRX -> SRR for 12 samples...
  [GSE202275] Resolving 12 SRX accessions via NCBI Entrez...
  [GSE202275] Resolved 12/12 SRX->SRR mappings.
  [GSE202275] Vehicle-only filter: 6 samples retained (VU846 arms excluded).
  [GSE202275] Written: ../config/coldata/GSE202275_coldata.csv (6 samples)
  [GSE202275] Condition counts: control=3, prion=3
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6525131 348.5    9928347 530.3  9928347 530.3
Vcells 11808358  90.1   21752374 166.0 17239063 131.6


Reading file....
Parsing....
Found 13 entities...
GPL19057 (1 of 14 entities)
GSM6107611 (2 of 14 entities)
GSM6107613 (3 of 14 entities)
GSM6107615 (4 of 14 entities)
GSM6107617 (5 of 14 entities)
GSM6107619 (6 of 14 entities)
GSM6107621 (7 of 14 entities)
GSM6107623 (8 of 14 entities)
GSM6107625 (9 of 14 entities)
GSM6107627 (10 of 14 entities)
GSM6107629 (11 of 14 entities)
GSM6107631 (12 of 14 entities)
GSM6107632 (13 of 14 entities)


In [28]:
%%R

# =============================================================================
# SECTION 5: VALIDATION AND SUMMARY
# =============================================================================

cat("\n=============================================================\n")
cat("  COLDATA BUILD COMPLETE — VALIDATION SUMMARY\n")
cat("=============================================================\n\n")

# --- 5.1 Cross-check: every SRR in sample_metadata.tsv should appear in coldata ---
# NOTE: sample_metadata.tsv contains ALL 867 downloaded SRRs, including samples
# that are intentionally excluded from the analysis (noCre, RiboTag IP, SARM1-/-,
# VU846-treated). The expected counts below reflect the VALID sample counts after
# exclusions, not the total downloaded counts.
#
# Expected valid sample counts (after intentional exclusions):
#   GSE149805: 176  (186 total - 10 noCre)
#   GSE198063:  58  (159 total - 101 RiboTag IP)
#   GSE124932:   8  (16 total - 8 SARM1-/-)
#   GSE202275:   6  (12 total - 6 VU846-treated)
#   All others: same as total (no exclusions)
#
# The cross-check below reports SRRs in sample_metadata.tsv but NOT in coldata.
# For the 4 studies above, these "missing" SRRs are INTENTIONALLY excluded.
# The script will print a NOTE (not WARNING) for intentionally excluded SRRs.

INTENTIONAL_EXCLUSIONS <- list(
  GSE149805 = c("SRR11684370","SRR11684371","SRR11684372","SRR11684373","SRR11684374",
                "SRR11684375","SRR11684376","SRR11684377","SRR11684378","SRR11684379"),
  GSE198063 = sample_meta %>% filter(GSE == "GSE198063") %>%
                pull(SRR) %>%
                setdiff(all_coldata[["GSE198063"]]$sample_id),
  GSE124932 = c("SRR8429841","SRR8429842","SRR8429843","SRR8429844",
                "SRR8429849","SRR8429850","SRR8429851","SRR8429852"),
  GSE202275 = sample_meta %>% filter(GSE == "GSE202275") %>%
                pull(SRR) %>%
                setdiff(all_coldata[["GSE202275"]]$sample_id)
)

cat("Cross-checking SRRs against sample_metadata.tsv...\n")

for (gse_id in names(all_coldata)) {
  cd       <- all_coldata[[gse_id]]
  expected <- sample_meta %>% filter(GSE == gse_id) %>% pull(SRR)
  in_cd    <- cd$sample_id
  missing  <- setdiff(expected, in_cd)
  extra    <- setdiff(in_cd, expected)

  # Separate intentional exclusions from unexpected missing
  intentional <- INTENTIONAL_EXCLUSIONS[[gse_id]]
  unexpected_missing <- setdiff(missing, intentional)
  noted_missing      <- intersect(missing, intentional)

  if (length(unexpected_missing) > 0) {
    cat(sprintf("  [%s] WARNING: %d SRRs unexpectedly missing from coldata: %s\n",
                gse_id, length(unexpected_missing),
                paste(head(unexpected_missing, 3), collapse = ", ")))
  }
  if (length(noted_missing) > 0) {
    cat(sprintf("  [%s] NOTE: %d SRRs intentionally excluded (see script header): %s...\n",
                gse_id, length(noted_missing),
                paste(head(noted_missing, 2), collapse = ", ")))
  }
  if (length(extra) > 0) {
    cat(sprintf("  [%s] NOTE: %d SRRs in coldata but NOT in sample_metadata.tsv: %s\n",
                gse_id, length(extra), paste(head(extra, 3), collapse = ", ")))
  }
  if (length(unexpected_missing) == 0 && length(extra) == 0) {
    cat(sprintf("  [%s] OK — %d valid samples, %d intentionally excluded.\n",
                gse_id, nrow(cd), length(noted_missing)))
  }
}

# --- 5.2 Check for NA conditions ---
cat("\nChecking for NA conditions...\n")
for (gse_id in names(all_coldata)) {
  cd <- all_coldata[[gse_id]]
  n_na <- sum(is.na(cd$condition))
  if (n_na > 0) {
    cat(sprintf("  [%s] WARNING: %d samples have NA condition — check parsing logic.\n",
                gse_id, n_na))
  }
}

# --- 5.3 Summary table ---
cat("\nFinal coldata summary:\n")
cat(sprintf("  %-12s  %-8s  %-6s  %-6s  %-40s\n",
            "Study", "Stratum", "prion", "ctrl", "Covariates"))
cat(sprintf("  %s\n", paste(rep("-", 80), collapse = "")))

for (gse_id in names(STUDY_STRATA)) {
  if (!gse_id %in% names(all_coldata)) {
    cat(sprintf("  %-12s  %-8s  MISSING\n", gse_id, STUDY_STRATA[gse_id]))
    next
  }
  cd      <- all_coldata[[gse_id]]
  n_prion <- sum(cd$condition == "prion",   na.rm = TRUE)
  n_ctrl  <- sum(cd$condition == "control", na.rm = TRUE)
  covs    <- setdiff(colnames(cd), c("sample_id", "files", "condition", "gsm_id"))
  cat(sprintf("  %-12s  %-8s  %-6d  %-6d  %s\n",
              gse_id, STUDY_STRATA[gse_id], n_prion, n_ctrl, paste(covs, collapse = ", ")))
}

# --- 5.4 Remind user about manual steps ---
cat("\n=============================================================\n")
cat("  NOTES BEFORE RUNNING 05_stream1_cns_meta_analysis.R\n")
cat("=============================================================\n")
cat("\n1. GSE214373 (human sCJD) — age column is NOW EMBEDDED (no manual step needed):\n")
cat("   Individual ages were retrieved from Supplementary Table 2 of:\n")
cat("     Dimitriadis et al. (2022) bioRxiv 2022.09.13.507650\n")
cat("   Ages are hard-coded in GSE214373_age_map above and written to coldata CSV.\n")
cat("   DESeq2 design: ~ age + sex + condition\n")
cat("   CAVEAT: age-condition r=-0.60 (p=0.024, N=14). Interpret condition\n")
cat("   estimates with caution; report this limitation in the manuscript.\n")
cat("\n2. Verify quant.sf paths:\n")
cat("   The 'files' column contains paths like:\n")
cat("   ../results/salmon_quant/<GSE>/<SRR>/quant.sf\n")
cat("   These paths will be validated by load_coldata() in 05_stream1_cns_meta_analysis.R.\n")
cat("   Ensure Salmon quantification (03_reads_quant.sh) is complete before running 05_.\n")
cat("\n3. Check condition assignments:\n")
cat("   Review the coldata CSVs to confirm condition labels are correct.\n")
cat("   Especially: GSE198063 (FFI+CJD both mapped to 'prion') and\n")
cat("   GSE90977 (long condition strings mapped to prion/control).\n")
cat("\n4. INTENTIONAL SAMPLE EXCLUSIONS (publication-level decisions):\n")
cat("   The following samples are in sample_metadata.tsv but NOT in coldata.\n")
cat("   This is CORRECT and expected — they are excluded for scientific reasons:\n")
cat("   - GSE149805: 10 noCre samples (SRR11684370-379) — no cell-type specificity\n")
cat("   - GSE198063: 101 RiboTag IP samples — incompatible with total RNA meta-analysis\n")
cat("   - GSE124932: 8 SARM1-/- samples — altered neurodegeneration pathway\n")
cat("   - GSE202275: 6 VU846-treated samples — drug intervention, not natural disease\n")
cat("   See script header (SAMPLE EXCLUSION RATIONALE) for full justification.\n")
cat("\n=============================================================\n")
cat("  04_build_coldata.R complete.\n")
cat("  Next step: run 05_stream1_cns_meta_analysis.R\n")
cat("=============================================================\n")



  COLDATA BUILD COMPLETE — VALIDATION SUMMARY

Cross-checking SRRs against sample_metadata.tsv...
  [GSE149805] NOTE: 10 SRRs intentionally excluded (see script header): SRR11684370, SRR11684371...
  [GSE149805] OK — 176 valid samples, 10 intentionally excluded.
  [GSE198063] NOTE: 101 SRRs intentionally excluded (see script header): SRR18268275, SRR18268276...
  [GSE198063] OK — 58 valid samples, 101 intentionally excluded.
  [GSE144738] OK — 137 valid samples, 0 intentionally excluded.
  [GSE189527] OK — 98 valid samples, 0 intentionally excluded.
  [GSE201249] OK — 50 valid samples, 0 intentionally excluded.
  [GSE210128] OK — 173 valid samples, 0 intentionally excluded.
  [GSE214373] OK — 14 valid samples, 0 intentionally excluded.
  [GSE90977] OK — 16 valid samples, 0 intentionally excluded.
  [GSE124932] NOTE: 8 SRRs intentionally excluded (see script header): SRR8429841, SRR8429842...
  [GSE124932] OK — 8 valid samples, 8 intentionally excluded.
  [GSE207251] OK — 6 valid sampl

##(R): Section 4B — Build Coldata: Stratum B & C Studies

**Purpose:** Build coldata for the remaining studies: GSE210128 (Stratum B, human brain sCJD), GSE214373 (Stratum C, human validation). Same approach as Section 4A but with human-specific metadata fields.

**Inputs:** GEO SOFT files (downloaded/cached), `sample_meta` from Section 2.

**Outputs:** `../config/coldata/coldata_GSE210128.tsv`, `coldata_GSE214373.tsv`.

**Key decisions:** Human studies use `tissue` and `disease_state` fields rather than the mouse-specific `brain_region` field.

In [8]:
%%R
# =============================================================================
# SECTION 4: PER-STUDY COLDATA CONSTRUCTION (Stratum B & C)
# =============================================================================

cat("\n=== GSE210128 ===\n")
gse210128 <- fetch_geo("GSE210128")
gsm_srr_210128 <- build_gsm_srr_map(gse210128, "GSE210128")

coldata_210128 <- lapply(names(gse210128$gsms), function(gsm_id) {
  gsm        <- gse210128$gsms[[gsm_id]]
  chars      <- parse_characteristics(gsm)
  gsm_title  <- str_to_lower(str_trim(GEOquery::Meta(gsm)$title[[1]]))
  srr        <- gsm_srr_210128$srr_id[gsm_srr_210128$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  treatment <- str_to_lower(chars[["treatment"]])
  condition <- case_when(
    str_detect(treatment, "nbh|control|mock|sham") ~ "control",
    str_detect(treatment, "rml|prion|scrapie")     ~ "prion",
    TRUE ~ NA_character_
  )

  tissue_raw <- str_to_lower(str_trim(chars[["tissue"]]))
  tissue <- case_when(
    str_detect(tissue_raw, "blood")   ~ "whole_blood",
    str_detect(tissue_raw, "spleen")  ~ "spleen",
    str_detect(tissue_raw, "muscle")  ~ "skeletal_muscle",
    TRUE ~ tissue_raw
  )

  tp_raw <- str_extract(gsm_title, "(\\d+|term)\\s*wpi")
  time_point_wpi <- if (!is.na(tp_raw)) {
    tp_num <- str_extract(tp_raw, "\\d+|term")
    if (tp_num == "term") 22L else as.integer(tp_num)
  } else {
    NA_integer_
  }

  cohort <- case_when(
    str_detect(gsm_title, "validation cohort") ~ "validation",
    str_detect(gsm_title, "main cohort")       ~ "main",
    TRUE ~ NA_character_
  )

  data.frame(
    sample_id      = srr,
    files          = quant_path(srr, "GSE210128"),
    condition      = condition,
    tissue         = tissue,
    time_point_wpi = time_point_wpi,
    cohort         = cohort,
    gsm_id         = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

cat("  GSE210128 sample breakdown:\n")
print(with(coldata_210128, table(tissue, time_point_wpi, condition)))

write_coldata(coldata_210128, "GSE210128")
all_coldata[["GSE210128"]] <- coldata_210128
rm(gse210128, gsm_srr_210128); gc()


cat("\n=== GSE214373 ===\n")
GSE214373_age_map <- c(
  GSM6604800 = 74L, GSM6604801 = 67L, GSM6604802 = 68L, GSM6604803 = 77L,
  GSM6604804 = 72L, GSM6604805 = 81L, GSM6604806 = 69L, GSM6604807 = 87L,
  GSM6604808 = 76L, GSM6604809 = 89L, GSM6604810 = 79L, GSM6604811 = 80L,
  GSM6604812 = 86L, GSM6604813 = 103L
)

gse214373 <- fetch_geo("GSE214373")
gsm_srr_214373 <- build_gsm_srr_map(gse214373, "GSE214373")

coldata_214373 <- lapply(names(gse214373$gsms), function(gsm_id) {
  gsm   <- gse214373$gsms[[gsm_id]]
  chars <- parse_characteristics(gsm)
  srr   <- gsm_srr_214373$srr_id[gsm_srr_214373$gsm_id == gsm_id]
  if (length(srr) == 0 || is.na(srr)) return(NULL)

  sample_title  <- GEOquery::Meta(gsm)$title[[1]]
  raw_condition <- str_to_lower(sample_title)
  condition <- case_when(
    str_detect(raw_condition, "scjd|cjd|prion") ~ "prion",
    str_detect(raw_condition, "control|normal")  ~ "control",
    TRUE ~ NA_character_
  )

  age <- GSE214373_age_map[gsm_id]
  if (is.na(age)) cat(sprintf("  [GSE214373] WARNING: no age found for %s\n", gsm_id))

  data.frame(
    sample_id = srr,
    files     = quant_path(srr, "GSE214373"),
    condition = condition,
    sex       = str_to_lower(str_trim(chars[["sex"]])),
    age       = as.integer(age),
    tissue    = "frontal_cortex",
    gsm_id    = gsm_id,
    stringsAsFactors = FALSE
  )
}) %>% bind_rows() %>% filter(!is.na(condition))

cat(sprintf("  [GSE214373] N=%d samples; age range %d–%d\n",
            nrow(coldata_214373), min(coldata_214373$age, na.rm = TRUE), max(coldata_214373$age, na.rm = TRUE)))

write_coldata(coldata_214373, "GSE214373")
all_coldata[["GSE214373"]] <- coldata_214373
rm(gse214373, gsm_srr_214373); gc()


=== GSE210128 ===
  Fetching GEO metadata for GSE210128 (SOFT file)...
  GSE210128: 173 GSM samples found.
  [GSE210128] Resolving GSM -> SRX -> SRR for 173 samples...
  [GSE210128] Resolving 173 SRX accessions via NCBI Entrez...
  [GSE210128] Resolved 173/173 SRX->SRR mappings.
  GSE210128 sample breakdown:
, , condition = control

                 time_point_wpi
tissue            4 8 12 14 16 18 20 22
  skeletal_muscle 3 6  6  3  6  3  6  6
  spleen          3 3  3  3  3  3  3  3
  whole_blood     3 3  2  3  3  3  3  3

, , condition = prion

                 time_point_wpi
tissue            4 8 12 14 16 18 20 22
  skeletal_muscle 3 6  6  3  6  3  6  5
  spleen          3 3  3  3  3  3  3  3
  whole_blood     3 3  4  3  3  3  3  3

  [GSE210128] Written: ../config/coldata/GSE210128_coldata.csv (173 samples)
  [GSE210128] Condition counts: control=86, prion=87

=== GSE214373 ===
  Fetching GEO metadata for GSE214373 (SOFT file)...
  GSE214373: 14 GSM samples found.
  [GSE214373] Reso

Reading file....
Parsing....
Found 174 entities...
GPL21103 (1 of 175 entities)
GSM6421805 (2 of 175 entities)
GSM6421806 (3 of 175 entities)
GSM6421807 (4 of 175 entities)
GSM6421808 (5 of 175 entities)
GSM6421809 (6 of 175 entities)
GSM6421810 (7 of 175 entities)
GSM6421811 (8 of 175 entities)
GSM6421812 (9 of 175 entities)
GSM6421813 (10 of 175 entities)
GSM6421817 (11 of 175 entities)
GSM6421818 (12 of 175 entities)
GSM6421819 (13 of 175 entities)
GSM6421823 (14 of 175 entities)
GSM6421824 (15 of 175 entities)
GSM6421828 (16 of 175 entities)
GSM6421829 (17 of 175 entities)
GSM6421830 (18 of 175 entities)
GSM6421831 (19 of 175 entities)
GSM6421834 (20 of 175 entities)
GSM6421835 (21 of 175 entities)
GSM6421836 (22 of 175 entities)
GSM6421837 (23 of 175 entities)
GSM6421838 (24 of 175 entities)
GSM6421839 (25 of 175 entities)
GSM6421840 (26 of 175 entities)
GSM6421841 (27 of 175 entities)
GSM6421842 (28 of 175 entities)
GSM6421845 (29 of 175 entities)
GSM6421846 (30 of 175 entities)


##(R): Section 5 — Validation & Final Output

**Purpose:** Validate all generated coldata files for completeness and consistency, then write a combined `all_coldata_summary.tsv` listing sample counts per study. Also validates that every SRR in `sample_metadata.tsv` has a matching Salmon quantification directory.

**Inputs:** All `coldata_GSE*.tsv` files written in Sections 4A–4B; `../results/salmon_quant/` directory.

**Outputs:** `../config/coldata/all_coldata_summary.tsv`; console report of missing/mismatched samples.

**Key decisions:** Any missing Salmon quant directory is reported as a warning (not an error) so the notebook completes even if a subset of samples failed quantification.

In [ ]:
%%R

# =============================================================================
# SECTION 5: VALIDATION AND SUMMARY
# =============================================================================

cat("\n=============================================================\n")
cat("  COLDATA BUILD COMPLETE — VALIDATION SUMMARY\n")
cat("=============================================================\n\n")

INTENTIONAL_EXCLUSIONS <- list(
  GSE149805 = c("SRR11684370","SRR11684371","SRR11684372","SRR11684373","SRR11684374",
                "SRR11684375","SRR11684376","SRR11684377","SRR11684378","SRR11684379"),
  GSE198063 = sample_meta %>% filter(GSE == "GSE198063") %>%
                pull(SRR) %>%
                setdiff(all_coldata[["GSE198063"]]$sample_id),
  GSE124932 = c("SRR8429841","SRR8429842","SRR8429843","SRR8429844",
                "SRR8429849","SRR8429850","SRR8429851","SRR8429852"),
  GSE202275 = sample_meta %>% filter(GSE == "GSE202275") %>%
                pull(SRR) %>%
                setdiff(all_coldata[["GSE202275"]]$sample_id)
)

for (gse_id in names(all_coldata)) {
  cd       <- all_coldata[[gse_id]]
  expected <- sample_meta %>% filter(GSE == gse_id) %>% pull(SRR)
  in_cd    <- cd$sample_id
  missing  <- setdiff(expected, in_cd)
  extra    <- setdiff(in_cd, expected)

  intentional <- INTENTIONAL_EXCLUSIONS[[gse_id]]
  unexpected_missing <- setdiff(missing, intentional)
  noted_missing      <- intersect(missing, intentional)

  if (length(unexpected_missing) > 0) cat(sprintf("  [%s] WARNING: %d SRRs missing.\n", gse_id, length(unexpected_missing)))
  if (length(noted_missing) > 0) cat(sprintf("  [%s] NOTE: %d SRRs intentionally excluded.\n", gse_id, length(noted_missing)))
  if (length(extra) > 0) cat(sprintf("  [%s] NOTE: %d SRRs in coldata but NOT in sample_metadata.tsv.\n", gse_id, length(extra)))
  if (length(unexpected_missing) == 0 && length(extra) == 0) cat(sprintf("  [%s] OK — %d valid samples, %d intentionally excluded.\n", gse_id, nrow(cd), length(noted_missing)))
}

cat("\nFinal coldata summary:\n")
cat(sprintf("  %-12s  %-8s  %-6s  %-6s  %-40s\n", "Study", "Stratum", "prion", "ctrl", "Covariates"))
cat(sprintf("  %s\n", paste(rep("-", 80), collapse = "")))

for (gse_id in names(STUDY_STRATA)) {
  if (!gse_id %in% names(all_coldata)) {
    cat(sprintf("  %-12s  %-8s  MISSING\n", gse_id, STUDY_STRATA[gse_id]))
    next
  }
  cd      <- all_coldata[[gse_id]]
  n_prion <- sum(cd$condition == "prion",   na.rm = TRUE)
  n_ctrl  <- sum(cd$condition == "control", na.rm = TRUE)
  covs    <- setdiff(colnames(cd), c("sample_id", "files", "condition", "gsm_id"))
  cat(sprintf("  %-12s  %-8s  %-6d  %-6d  %s\n",
              gse_id, STUDY_STRATA[gse_id], n_prion, n_ctrl, paste(covs, collapse = ", ")))
}

cat("\n=============================================================\n")
cat("  04_build_coldata.R complete.\n")
cat("  Your metadata is perfectly synchronized for the rest of your pipeline.\n")
cat("=============================================================\n")


  COLDATA BUILD COMPLETE — VALIDATION SUMMARY

  [GSE149805] NOTE: 10 SRRs intentionally excluded.
  [GSE149805] OK — 176 valid samples, 10 intentionally excluded.
  [GSE198063] NOTE: 101 SRRs intentionally excluded.
  [GSE198063] OK — 58 valid samples, 101 intentionally excluded.
  [GSE144738] OK — 137 valid samples, 0 intentionally excluded.
  [GSE189527] OK — 98 valid samples, 0 intentionally excluded.
  [GSE201249] OK — 50 valid samples, 0 intentionally excluded.
  [GSE210128] OK — 173 valid samples, 0 intentionally excluded.
  [GSE214373] OK — 14 valid samples, 0 intentionally excluded.
  [GSE90977] OK — 16 valid samples, 0 intentionally excluded.
  [GSE124932] NOTE: 8 SRRs intentionally excluded.
  [GSE124932] OK — 8 valid samples, 8 intentionally excluded.
  [GSE207251] OK — 6 valid samples, 0 intentionally excluded.
  [GSE202275] NOTE: 6 SRRs intentionally excluded.
  [GSE202275] OK — 6 valid samples, 6 intentionally excluded.

Final coldata summary:
  Study         Stratum   